# Normalize sqY2H consensus scores

This script applies various QA/QC steps and then computes a normalized interaction score. For QA/QC we:

1.   Check whether 6 media controls on each mega-plate meet various condition-specific thresholds
2.   Check for autoactivation mating failures
3.   Adjust consensus score based on autoactivation score and filter if score invalid

In [53]:
import polars as pl 
import pandas as pd
import numpy as np

In [54]:
inputs_dir = "../1_inputs/sqY2H"
outputs_dir = "../3_outputs/sqY2H"
meta_inputs = "../../../1_allele_collection/1_inputs"
meta_outputs = "../../../1_allele_collection/3_outputs"

scoring_conditions = ['LWH1', 'LWH10', 'LWH25', 'LWA']
max_score = 4
aa_id = '0.0'
rrs_id = 999999.0

In [55]:
def df_to_dict_variants(df):
    if isinstance(df, pl.DataFrame):
        df = df.to_pandas()
        
    if 'media_control' in df.columns:
        df = df[df['media_control'] == False].copy()
        
    scoring_conditions = ['LWH1', 'LWH10', 'LWH25', 'LWA']
        
    if 'condition_name' in df.columns:
        df = df[df['condition_name'].isin(scoring_conditions)].copy()
    
    cols_to_keep = ['db_orf_id', 'ad_orf_id', 'db_mut_id', 'retest_batch']

    df = df[cols_to_keep].copy()

    df[['db_orf_id', 'ad_orf_id', 'db_mut_id']] = df[['db_orf_id', 'ad_orf_id', 'db_mut_id']].map(float).map(int)
        
    def filter_df(df):
        df = df[(df.ad_orf_id != 0) 
                & (df.db_mut_id != 999999)].copy()
        
        return df
    
    df = filter_df(df)
    
    df_dict = {}
    
    assay_array = ["VUSAPWT1B1","VUSAPWT1B2","VUSAPWT2B1","VUSAPWT6B1"]
    
    for retest_batch in assay_array:
        df_assay = df[df['retest_batch'] == retest_batch]
        df_dict[retest_batch] = {}
        for db_orf_id, df_db in df_assay.sort_values('db_orf_id').groupby('db_orf_id'):
            df_dict[retest_batch][db_orf_id] = {}
            for i, row in df_db.sort_values('db_mut_id').iterrows():
                db_mut_id = row['db_mut_id']
                ad_orf_id = row['ad_orf_id']
                if db_mut_id not in df_dict[retest_batch][db_orf_id].keys():
                    df_dict[retest_batch][db_orf_id][db_mut_id] = {ad_orf_id}
                else:
                    df_dict[retest_batch][db_orf_id][db_mut_id].add(ad_orf_id)
                    
    return df_dict

## Apply all QA/QC thresholds

In [56]:
# get media control thresholds
media_score_thresholds = pd.read_csv(f"{inputs_dir}/media_control_score_thresholds.csv")
media_score_thresholds.rename(columns={'control_96': 'scoring_pos'}, inplace=True)

In [99]:
scores = pd.read_csv(f"{outputs_dir}/2_raw_consensus_scores.csv") 

# Split into media controls
media_ctrls = scores[scores['media_control']]
scores = scores[~scores['media_control']]

# Check media controls
media_ctrls = media_ctrls[['db_orf_id', 'condition_name', 'retest_batch', 'consensus_score']]
media_ctrls[['scoring_pos', 'retest_pla']] = media_ctrls['db_orf_id'].str.split('_', n=1, expand=True)
media_ctrls = media_ctrls.merge(media_score_thresholds, on=['scoring_pos', 'condition_name'])
media_ctrls['score_in_bounds'] = media_ctrls['consensus_score'].between(media_ctrls['lower_bound'], media_ctrls['upper_bound'], inclusive='both')

# replace failure due to variable scoring (99999) with NaN to group with other technical failures
scores['consensus_score'] = scores['consensus_score'].replace(99999, np.nan)

# add IDs for groups used for filtering
scores = pl.DataFrame(scores)

scores = scores.with_columns(
    pl.col("retest_batch").str.extract(r"VUSAPWT(\d)B\d+", 1).alias("assay_id")
)

scores = scores.with_columns(
    pl.concat_str([
            pl.col('db_orf_id').cast(pl.String),
            pl.col('db_mut_id').cast(pl.String)
    ], separator="_").alias("db_group"),
    pl.concat_str([
            pl.col('db_orf_id').cast(pl.String),
            pl.col('db_mut_id').cast(pl.String),
            pl.col('condition_name').cast(pl.String),
            pl.col('retest_batch').cast(pl.String)
    ], separator="_").alias("db_condition_group"),
    pl.concat_str([
            pl.col('ad_orf_id').cast(pl.String),
            pl.col('db_orf_id').cast(pl.String),
            pl.col('db_mut_id').cast(pl.String),
            pl.col('retest_batch').cast(pl.String)
    ], separator="_").alias("ppi_group")
)

dict_before_qc = df_to_dict_variants(scores)

In [58]:
scores

ad_orf_id,ad_mut_id,db_orf_id,db_mut_id,condition_name,retest_batch,media_control,consensus_score,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0,assay_id,db_group,db_condition_group,ppi_group
str,f64,str,f64,str,str,bool,f64,str,str,str,str,str,str,str,f64,f64,str,str,str,str,str
"""0.0""",null,"""53657.0""",9.99999e5,"""LWA""","""VUSAPWT2B1""",false,0.0,"""empty_ad""",null,null,null,null,null,null,null,null,null,"""2""","""53657.0_999999.0""","""53657.0_999999.0_LWA_VUSAPWT2B…","""0.0_53657.0_999999.0_VUSAPWT2B…"
"""6552.0""",null,"""2713.0""",200868.0,"""LWH25""","""VUSAPWT1B1""",false,2.0,"""MAGEA8""","""MLH1""","""Asp485Glu""","""1455T>A""","""ENSG00000076242""","""CEGS2""",null,null,null,"""MLH1_Asp485Glu""","""1""","""2713.0_200868.0""","""2713.0_200868.0_LWH25_VUSAPWT1…","""6552.0_2713.0_200868.0_VUSAPWT…"
"""7845.0""",null,"""7038.0""",0.0,"""LWH10""","""VUSAPWT1B2""",false,2.0,"""AGTRAP""","""DIABLO""",null,null,"""ENSG00000184047""",null,null,null,null,"""DIABLO""","""1""","""7038.0_0.0""","""7038.0_0.0_LWH10_VUSAPWT1B2""","""7845.0_7038.0_0.0_VUSAPWT1B2"""
"""11621.0""",null,"""4062.0""",201432.0,"""LWH10""","""VUSAPWT6B1""",false,0.0,"""MTIF3""","""PLP1""","""Val184Phe""","""550G>T""","""ENSG00000123560""","""CEGS2""","""1_Pathogenic""",null,2.0,"""PLP1_Val184Phe""","""6""","""4062.0_201432.0""","""4062.0_201432.0_LWH10_VUSAPWT6…","""11621.0_4062.0_201432.0_VUSAPW…"
"""13569.0""",null,"""12568.0""",0.0,"""LW""","""VUSAPWT1B2""",false,4.0,"""CTIF""","""ATPAF2""",null,null,"""ENSG00000171953""",null,null,null,null,"""ATPAF2""","""1""","""12568.0_0.0""","""12568.0_0.0_LW_VUSAPWT1B2""","""13569.0_12568.0_0.0_VUSAPWT1B2"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""11671.0""",null,"""2713.0""",704.0,"""LW""","""VUSAPWT2B1""",false,4.0,"""FAN1""","""MLH1""","""His318Gln""","""954C>A""","""ENSG00000076242""","""Edgotyping3""","""4_VUS""",0.000002,2.0,"""MLH1_His318Gln""","""2""","""2713.0_704.0""","""2713.0_704.0_LW_VUSAPWT2B1""","""11671.0_2713.0_704.0_VUSAPWT2B…"
"""11256.0""",null,"""54308.0""",3030.0,"""LWH10""","""VUSAPWT1B1""",false,1.0,"""RABEP1""","""PKP2""","""Lys724Thr""","""2171A>C""","""ENSG00000057294""","""Edgotyping3""","""3_Conflicting""",null,1.0,"""PKP2_Lys724Thr""","""1""","""54308.0_3030.0""","""54308.0_3030.0_LWH10_VUSAPWT1B…","""11256.0_54308.0_3030.0_VUSAPWT…"
"""7845.0""",null,"""13295.0""",218062.0,"""LWA""","""VUSAPWT1B1""",false,1.0,"""AGTRAP""","""TTPA""","""Arg192His""","""575G>A""","""ENSG00000137561""","""CEGS2""","""3_Conflicting""",null,1.0,"""TTPA_Arg192His""","""1""","""13295.0_218062.0""","""13295.0_218062.0_LWA_VUSAPWT1B…","""7845.0_13295.0_218062.0_VUSAPW…"


In [59]:
# Remove all db groups with the same allele if there was an autoactivation mating failure
aa_db_group_mating_failures = (scores.filter(pl.col("ad_orf_id") == aa_id)
                              .filter(pl.col("condition_name") == "LW")
                              .filter(pl.col("consensus_score") <= 2)
                              .select("db_group")
                              .to_series().unique().to_list())
scores = scores.filter(~pl.col("db_group").is_in(aa_db_group_mating_failures))

dict_after_db_groups = df_to_dict_variants(scores)

# Remove all db-condition groups with the same allele if there was an autoactivation mating failure
aa_db_group_tech_failures = (scores.filter(pl.col("ad_orf_id") == aa_id)
                                   .filter(pl.col("consensus_score").is_nan())
                                   .select("db_condition_group")
                                   .to_series().unique().to_list())
scores = scores.filter(~pl.col("db_condition_group").is_in(aa_db_group_tech_failures))

dict_after_db_conditions = df_to_dict_variants(scores)

# remove mating failures - condition 1 shows no or little yeast growth
mating_failures = (scores.filter(pl.col("condition_name") == "LW")
                         .filter(pl.col("consensus_score") <= 2)
                         .select("ppi_group").to_series().unique().to_list())
scores = scores.filter(~pl.col("ppi_group").is_in(mating_failures))

dict_after_mating = df_to_dict_variants(scores)


# Adjust consensus score based on autoactivation score
aa_scores = scores.filter(pl.col("ad_orf_id") == aa_id).select([
    "db_condition_group", "consensus_score"
]).rename({"consensus_score": "aa_score"})

# Remove autoactivation rows from the main scores df
# Add the aa_score to the dataframe, merged on the condition-specific db allele
scores = scores.filter(pl.col("ad_orf_id") != aa_id).join(aa_scores, on="db_condition_group")

scores = scores.with_columns(
    (pl.col("consensus_score") - pl.col("aa_score")).alias("adjusted_score")
).filter(
    ~((pl.col("adjusted_score") <= 0) & (pl.col("aa_score") > 0))
)

dict_after_aa = df_to_dict_variants(scores)

In [60]:
print(aa_db_group_mating_failures)
print(len(aa_db_group_mating_failures))
print(aa_db_group_tech_failures)
print(len(aa_db_group_tech_failures))
print(mating_failures)
print(len(mating_failures))

['11435.0_999999.0', '13571.0_218098.0', '281.0_29.0', '100016069.0_2637.0', '55222.0_999999.0', '8416.0_216642.0', '636.0_999999.0', '100016069.0_2624.0', '100016069.0_2620.0', '2920.0_214415.0', '100016069.0_2629.0', '100016069.0_2638.0']
12
[]
0
['9230.0_2920.0_214419.0_VUSAPWT2B1', '54043.0_13295.0_218057.0_VUSAPWT2B1', '3068.0_4004.0_201387.0_VUSAPWT2B1', '10677.0_7675.0_0.0_VUSAPWT2B1', '3068.0_4004.0_0.0_VUSAPWT2B1', '3990.0_71001.0_205674.0_VUSAPWT2B1', '53030.0_53030.0_0.0_VUSAPWT2B1', '824.0_7147.0_0.0_VUSAPWT1B2', '9230.0_2920.0_214422.0_VUSAPWT2B1', '2758.0_5723.0_0.0_VUSAPWT2B1', '100062710.0_11556.0_999999.0_VUSAPWT1B1', '14596.0_4062.0_201432.0_VUSAPWT2B1', '7881.0_53030.0_218690.0_VUSAPWT1B2', '53536.0_4062.0_201434.0_VUSAPWT6B1', '54341.0_2713.0_704.0_VUSAPWT1B1', '9241.0_5424.0_201909.0_VUSAPWT2B1', '14596.0_7323.0_0.0_VUSAPWT2B1', '14596.0_7323.0_216369.0_VUSAPWT2B1', '4761.0_2713.0_461.0_VUSAPWT1B1', '14997.0_71075.0_0.0_VUSAPWT2B1', '2194.0_5933.0_0.0_VUSAPWT2B1', 

In [61]:
scores

ad_orf_id,ad_mut_id,db_orf_id,db_mut_id,condition_name,retest_batch,media_control,consensus_score,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0,assay_id,db_group,db_condition_group,ppi_group,aa_score,adjusted_score
str,f64,str,f64,str,str,bool,f64,str,str,str,str,str,str,str,f64,f64,str,str,str,str,str,f64,f64
"""6552.0""",null,"""2713.0""",200868.0,"""LWH25""","""VUSAPWT1B1""",false,2.0,"""MAGEA8""","""MLH1""","""Asp485Glu""","""1455T>A""","""ENSG00000076242""","""CEGS2""",null,null,null,"""MLH1_Asp485Glu""","""1""","""2713.0_200868.0""","""2713.0_200868.0_LWH25_VUSAPWT1…","""6552.0_2713.0_200868.0_VUSAPWT…",0.0,2.0
"""7845.0""",null,"""7038.0""",0.0,"""LWH10""","""VUSAPWT1B2""",false,2.0,"""AGTRAP""","""DIABLO""",null,null,"""ENSG00000184047""",null,null,null,null,"""DIABLO""","""1""","""7038.0_0.0""","""7038.0_0.0_LWH10_VUSAPWT1B2""","""7845.0_7038.0_0.0_VUSAPWT1B2""",0.0,2.0
"""11621.0""",null,"""4062.0""",201432.0,"""LWH10""","""VUSAPWT6B1""",false,0.0,"""MTIF3""","""PLP1""","""Val184Phe""","""550G>T""","""ENSG00000123560""","""CEGS2""","""1_Pathogenic""",null,2.0,"""PLP1_Val184Phe""","""6""","""4062.0_201432.0""","""4062.0_201432.0_LWH10_VUSAPWT6…","""11621.0_4062.0_201432.0_VUSAPW…",0.0,0.0
"""12626.0""",null,"""9905.0""",0.0,"""LWH10""","""VUSAPWT1B2""",false,0.0,"""CNTROB""","""PNKP""",null,null,"""ENSG00000039650""",null,null,null,null,"""PNKP""","""1""","""9905.0_0.0""","""9905.0_0.0_LWH10_VUSAPWT1B2""","""12626.0_9905.0_0.0_VUSAPWT1B2""",0.0,0.0
"""2615.0""",null,"""7652.0""",1807.0,"""LWH25""","""VUSAPWT2B1""",false,0.0,"""STX19""","""STXBP1""","""His245Arg""","""734A>G""","""ENSG00000136854""","""Edgotyping3""","""1_Pathogenic""",null,2.0,"""STXBP1_His245Arg""","""2""","""7652.0_1807.0""","""7652.0_1807.0_LWH25_VUSAPWT2B1""","""2615.0_7652.0_1807.0_VUSAPWT2B…",0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""6534.0""",null,"""8420.0""",216659.0,"""LWH1""","""VUSAPWT1B2""",false,2.0,"""BANP""","""ALAS2""","""Gly254Ser""","""760G>A""","""ENSG00000158578""","""CEGS2""","""1_Pathogenic""",null,null,"""ALAS2_Gly254Ser""","""1""","""8420.0_216659.0""","""8420.0_216659.0_LWH1_VUSAPWT1B…","""6534.0_8420.0_216659.0_VUSAPWT…",0.0,2.0
"""11256.0""",null,"""54308.0""",3030.0,"""LWH10""","""VUSAPWT1B1""",false,1.0,"""RABEP1""","""PKP2""","""Lys724Thr""","""2171A>C""","""ENSG00000057294""","""Edgotyping3""","""3_Conflicting""",null,1.0,"""PKP2_Lys724Thr""","""1""","""54308.0_3030.0""","""54308.0_3030.0_LWH10_VUSAPWT1B…","""11256.0_54308.0_3030.0_VUSAPWT…",0.0,1.0
"""7845.0""",null,"""13295.0""",218062.0,"""LWA""","""VUSAPWT1B1""",false,1.0,"""AGTRAP""","""TTPA""","""Arg192His""","""575G>A""","""ENSG00000137561""","""CEGS2""","""3_Conflicting""",null,1.0,"""TTPA_Arg192His""","""1""","""13295.0_218062.0""","""13295.0_218062.0_LWA_VUSAPWT1B…","""7845.0_13295.0_218062.0_VUSAPW…",0.0,1.0


In [62]:
# split rrs & wt scores
rrs_scores = scores.filter(pl.col('db_mut_id') == rrs_id).filter(pl.col("condition_name") == "LWH1")
rrs_count_batch = rrs_scores.group_by(["retest_batch", "adjusted_score"]).agg(pl.len()).pivot(
    values="len", on="adjusted_score", index="retest_batch"
).select(["retest_batch", "0.0", "1.0", "2.0", "3.0", "4.0", "null"])
print(rrs_count_batch)

wt_scores = scores.filter(pl.col('db_mut_id') == 0.0).filter(pl.col("condition_name") == "LWH1")
wt_count_batch = wt_scores.group_by(["retest_batch", "adjusted_score"]).agg(pl.len()).pivot(
    values="len", on="adjusted_score", index="retest_batch"
).select(["retest_batch", "0.0", "1.0", "2.0", "3.0", "4.0", "null"])
print(wt_count_batch)

shape: (4, 7)
┌──────────────┬─────┬─────┬─────┬──────┬──────┬──────┐
│ retest_batch ┆ 0.0 ┆ 1.0 ┆ 2.0 ┆ 3.0  ┆ 4.0  ┆ null │
│ ---          ┆ --- ┆ --- ┆ --- ┆ ---  ┆ ---  ┆ ---  │
│ str          ┆ u32 ┆ u32 ┆ u32 ┆ u32  ┆ u32  ┆ u32  │
╞══════════════╪═════╪═════╪═════╪══════╪══════╪══════╡
│ VUSAPWT6B1   ┆ 165 ┆ 2   ┆ 1   ┆ 1    ┆ 2    ┆ null │
│ VUSAPWT1B2   ┆ 147 ┆ 4   ┆ 3   ┆ null ┆ null ┆ 3    │
│ VUSAPWT2B1   ┆ 165 ┆ 2   ┆ 3   ┆ 4    ┆ null ┆ null │
│ VUSAPWT1B1   ┆ 153 ┆ 1   ┆ 3   ┆ 2    ┆ 2    ┆ null │
└──────────────┴─────┴─────┴─────┴──────┴──────┴──────┘
shape: (4, 7)
┌──────────────┬─────┬─────┬─────┬─────┬─────┬──────┐
│ retest_batch ┆ 0.0 ┆ 1.0 ┆ 2.0 ┆ 3.0 ┆ 4.0 ┆ null │
│ ---          ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ ---  │
│ str          ┆ u32 ┆ u32 ┆ u32 ┆ u32 ┆ u32 ┆ u32  │
╞══════════════╪═════╪═════╪═════╪═════╪═════╪══════╡
│ VUSAPWT1B1   ┆ 3   ┆ 4   ┆ 13  ┆ 19  ┆ 65  ┆ null │
│ VUSAPWT1B2   ┆ 144 ┆ 19  ┆ 70  ┆ 215 ┆ 446 ┆ 3    │
│ VUSAPWT2B1   ┆ 45  ┆ 11  ┆ 40  ┆

In [63]:
# Keep only scores that aren't RRS pairs
scores = scores.filter(pl.col('db_mut_id') != rrs_id)

scores = scores.with_columns(
    pl.col("retest_batch").str.extract(r"VUSAPWT(\d)B\d+", 1).alias("assay_id")
)

In [64]:
# read in sequence confirmation files
ad_seq = pl.read_csv(f"{inputs_dir}/swimseq_confirmation/vus_ad_interactors.csv").rename({
    "swim_seq_validated": "seq_ad"
}).select([
    "ad_orf_id", "assay_id", "seq_ad"
])

db_var_seq = pl.read_csv(f"{inputs_dir}/swimseq_confirmation/vus_db_alleles.csv").rename({
    "orf_id": "db_orf_id",
    "mut_id": "db_mut_id",
    "swim_seq_validated": "seq_db"
}).select(["db_orf_id", "db_mut_id", "seq_db"])

db_ref_seq = pl.read_csv(f"{inputs_dir}/swimseq_confirmation/wt_disease_nodes_summary.csv").rename({
    "disease_orf_id": "db_orf_id",
    "swim_seq_validated": "seq_db"
}).select(["db_orf_id", "seq_db"]).with_columns(
    pl.lit(0).cast(pl.Int64).alias("db_mut_id")
).select(["db_orf_id", "db_mut_id", "seq_db"])

db_seq = pl.concat([db_var_seq, db_ref_seq])
# print(db_seq)

## Add an extra QC step to check the sequence confirmation results
seq_confirm_res_df = pd.read_table(f"{meta_inputs}/seq_confirmation_results/VarChampSeqConfirmationResult.tsv")
# with pd.option_context('display.max_columns', None):
#     display(seq_confirm_res_df)

db_seq = pl.DataFrame(
    pd.merge(db_seq.to_pandas(), 
            seq_confirm_res_df[
                ["orf_id_wt","mutation_id_old","db_sequenced","db_sequence_confirmation_class"] ##"symbol","aa_change","nt_change","ccsb_mutation_id"
            ].rename({"orf_id_wt": "db_orf_id",
                      "mutation_id_old": "db_mut_id"}, axis=1), 
        on=["db_orf_id","db_mut_id"],
        how="left"
    )
)
print(db_seq)
# print(db_seq["db_sequenced"].unique())
# print(db_seq.filter(pl.col("db_sequenced")==0))

shape: (2_887, 5)
┌───────────┬───────────┬────────┬──────────────┬────────────────────────────────┐
│ db_orf_id ┆ db_mut_id ┆ seq_db ┆ db_sequenced ┆ db_sequence_confirmation_class │
│ ---       ┆ ---       ┆ ---    ┆ ---          ┆ ---                            │
│ i64       ┆ i64       ┆ str    ┆ f64          ┆ f64                            │
╞═══════════╪═══════════╪════════╪══════════════╪════════════════════════════════╡
│ 281       ┆ 727       ┆ NULL   ┆ 1.0          ┆ 1.0                            │
│ 281       ┆ 739       ┆ 1      ┆ 1.0          ┆ 1.0                            │
│ 281       ┆ 13        ┆ 1      ┆ 1.0          ┆ 1.0                            │
│ 281       ┆ 744       ┆ 1      ┆ 1.0          ┆ 1.0                            │
│ 281       ┆ 763       ┆ 1      ┆ 1.0          ┆ 1.0                            │
│ …         ┆ …         ┆ …      ┆ …            ┆ …                              │
│ 8514      ┆ 0         ┆ 1      ┆ null         ┆ null               

In [65]:
print(scores.shape)

(40034, 24)


In [66]:
# cast all to same format
scores = scores.with_columns(
    pl.col("ad_orf_id").cast(pl.Float64).cast(pl.Int64).alias("ad_orf_id"),
    pl.col("db_orf_id").cast(pl.Float64).cast(pl.Int64).alias("db_orf_id"),
    pl.col("db_mut_id").cast(pl.Float64).cast(pl.Int64).alias("db_mut_id"),
    pl.col("assay_id").cast(pl.String).alias("assay_id"),
)

db_seq = db_seq.with_columns(
    pl.col("db_orf_id").cast(pl.Float64).cast(pl.Int64).alias("db_orf_id"),
    pl.col("db_mut_id").cast(pl.Float64).cast(pl.Int64).alias("db_mut_id"),
    pl.col("seq_db").cast(pl.String).alias("seq_db")
)

ad_seq = ad_seq.with_columns(
    pl.col("ad_orf_id").cast(pl.Float64).cast(pl.Int64).alias("ad_orf_id"),
    pl.col("seq_ad").cast(pl.String).alias("seq_ad"),
    pl.col("assay_id").cast(pl.String).alias("assay_id"),
)

scores = scores.join(ad_seq, on=["ad_orf_id", "assay_id"], how="left")
scores = scores.join(db_seq, on=["db_orf_id", "db_mut_id"], how="left")

# print(scores.filter(pl.col("db_mut_id")==0))

## Filter to only keep interactions where both ad and db are confirmed
## And also confirmed by the sequence confirmation results
valid_db_sequence_confirmation_class = [1., 2., 7., 99.]
scores = scores.filter(
    (pl.col("seq_ad") == "1") & (pl.col("seq_db") == "1") & (((pl.col("db_sequence_confirmation_class").is_in(valid_db_sequence_confirmation_class)) & (pl.col("db_sequenced").is_not_null())) | (pl.col("db_mut_id")==0))
)

## After sequence confirmation, we lost 25521 - 21895 = 3626 interactions
# print(scores.shape)
print(scores)

shape: (21_895, 28)
┌───────────┬───────────┬───────────┬───────────┬───┬────────┬────────┬──────────────┬─────────────┐
│ ad_orf_id ┆ ad_mut_id ┆ db_orf_id ┆ db_mut_id ┆ … ┆ seq_ad ┆ seq_db ┆ db_sequenced ┆ db_sequence │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---    ┆ ---    ┆ ---          ┆ _confirmati │
│ i64       ┆ f64       ┆ i64       ┆ i64       ┆   ┆ str    ┆ str    ┆ f64          ┆ on_class    │
│           ┆           ┆           ┆           ┆   ┆        ┆        ┆              ┆ ---         │
│           ┆           ┆           ┆           ┆   ┆        ┆        ┆              ┆ f64         │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪════════╪════════╪══════════════╪═════════════╡
│ 6552      ┆ null      ┆ 2713      ┆ 200868    ┆ … ┆ 1      ┆ 1      ┆ 1.0          ┆ 1.0         │
│ 7845      ┆ null      ┆ 7038      ┆ 0         ┆ … ┆ 1      ┆ 1      ┆ null         ┆ null        │
│ 11621     ┆ null      ┆ 4062      ┆ 201432    ┆ … ┆ 1      ┆ 1      ┆

In [67]:
dict_after_seq_1 = df_to_dict_variants(scores)

In [68]:
# write out results
scores.write_csv(f"{outputs_dir}/3_normalized_qaqc_consensus_scores.csv")

## Calculate normalized scores

In [69]:
# prepare data to calculate normalized scores
normalized_scores = scores.filter(pl.col("condition_name").is_in(scoring_conditions))

normalized_scores = normalized_scores.group_by([
    'ad_orf_id', 'db_orf_id', 'db_mut_id', 'retest_batch'
]).agg(
    (pl.col("adjusted_score").sum() / (len(scoring_conditions)*max_score)).alias("normalized_score")
)

normalized_scores = normalized_scores.with_columns(
    pl.col("retest_batch").str.extract(r"VUSAPWT(\d)B\d+", 1).alias("assay_id")
)
normalized_scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,normalized_score,assay_id
i64,i64,i64,str,f64,str
2860,7982,202790,"""VUSAPWT1B2""",1.0,"""1"""
55237,70497,3805,"""VUSAPWT1B1""",0.0,"""1"""
4623,11443,0,"""VUSAPWT1B2""",0.0,"""1"""
6281,100016069,2639,"""VUSAPWT6B1""",0.0,"""6"""
56141,4870,215286,"""VUSAPWT1B2""",0.5625,"""1"""
…,…,…,…,…,…
54668,70497,3851,"""VUSAPWT1B1""",0.25,"""1"""
4636,5904,0,"""VUSAPWT1B2""",0.625,"""1"""
11621,4062,201430,"""VUSAPWT6B1""",0.0,"""6"""


In [70]:
# cast all to same format
normalized_scores = normalized_scores.with_columns(
    pl.col("ad_orf_id").cast(pl.Float64).cast(pl.Int64).alias("ad_orf_id"),
    pl.col("db_orf_id").cast(pl.Float64).cast(pl.Int64).alias("db_orf_id"),
    pl.col("db_mut_id").cast(pl.Float64).cast(pl.Int64).alias("db_mut_id"),
    pl.col("assay_id").cast(pl.String).alias("assay_id"),
)

db_seq = db_seq.with_columns(
    pl.col("db_orf_id").cast(pl.Float64).cast(pl.Int64).alias("db_orf_id"),
    pl.col("db_mut_id").cast(pl.Float64).cast(pl.Int64).alias("db_mut_id"),
    pl.col("seq_db").cast(pl.String).alias("seq_db")
)

ad_seq = ad_seq.with_columns(
    pl.col("ad_orf_id").cast(pl.Float64).cast(pl.Int64).alias("ad_orf_id"),
    pl.col("seq_ad").cast(pl.String).alias("seq_ad"),
    pl.col("assay_id").cast(pl.String).alias("assay_id"),
)

normalized_scores = normalized_scores.join(ad_seq, on=["ad_orf_id", "assay_id"], how="left")
normalized_scores = normalized_scores.join(db_seq, on=["db_orf_id", "db_mut_id"], how="left")


# Filter to only keep interactions where both ad and db are confirmed
normalized_scores = normalized_scores.filter(
    (pl.col("seq_ad") == "1") & (pl.col("seq_db") == "1")
)
normalized_scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class
i64,i64,i64,str,f64,str,str,str,f64,f64
2860,7982,202790,"""VUSAPWT1B2""",1.0,"""1""","""1""","""1""",1.0,1.0
55237,70497,3805,"""VUSAPWT1B1""",0.0,"""1""","""1""","""1""",1.0,1.0
4623,11443,0,"""VUSAPWT1B2""",0.0,"""1""","""1""","""1""",null,null
6281,100016069,2639,"""VUSAPWT6B1""",0.0,"""6""","""1""","""1""",1.0,2.0
56141,4870,215286,"""VUSAPWT1B2""",0.5625,"""1""","""1""","""1""",1.0,1.0
…,…,…,…,…,…,…,…,…,…
54668,70497,3851,"""VUSAPWT1B1""",0.25,"""1""","""1""","""1""",1.0,1.0
4636,5904,0,"""VUSAPWT1B2""",0.625,"""1""","""1""","""1""",null,null
11621,4062,201430,"""VUSAPWT6B1""",0.0,"""6""","""1""","""1""",1.0,1.0


In [71]:
dict_after_seq_2 = df_to_dict_variants(normalized_scores)

In [72]:
# # map ad orfs ids to symbol

# horfeome_df = pd.read_csv(f"{meta_inputs}/horfeome_81_91_union.tsv", sep="\t")
# orf_to_symbol_dict = horfeome_df.set_index('orf_id')['symbol'].to_dict()

# # add a key for 0 as empty_AD
# orf_to_symbol_dict[0] = 'empty_AD'

# # update values where orfs are mapped to multiple symbols to only retain one previously used in the lab
# single_symbol_mapping_dict = {
#     1063: 'SMN2',
#     13954: 'AC005833.1',
#     55237: 'NBPF19',
#     2658: 'ZNF559-ZNF177',
#     1940: 'RBMY1F',
#     56585: 'CKMT1A',
#     2794: 'HBA1',
#     978: 'MAGEA2B',
#     70851: 'FAM25G',
#     3824: 'CKMT1A',
#     13813: 'CRYAA',
#     66: 'SMN1',
#     55944: 'KRTAP2-4',
#     13597: 'FAM187A',
#     56508: 'PRR20E',
#     100066377: 'TBC1D3G'
#     }


# for k, v in single_symbol_mapping_dict.items():
#     orf_to_symbol_dict[k] = v


# # apply mapping
# normalized_scores = normalized_scores.with_columns(
#     pl.col("ad_orf_id").replace_strict(orf_to_symbol_dict, default=None).alias("ad_symbol")
# )

In [73]:
# map ad_orf_id to ad_symbol
mapping_file = pd.read_csv(f"{inputs_dir}/cp_files/mapping_y2h.csv")
mapping_file.dropna(subset=['ad_orf_id'], inplace=True)
mapping_dict = mapping_file.set_index('ad_orf_id')['ad_symbol'].to_dict()

normalized_scores = normalized_scores.with_columns(
    pl.col("ad_orf_id").replace_strict(mapping_dict, default=None).alias("ad_symbol")
)
normalized_scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol
i64,i64,i64,str,f64,str,str,str,f64,f64,str
2860,7982,202790,"""VUSAPWT1B2""",1.0,"""1""","""1""","""1""",1.0,1.0,"""PRTFDC1"""
55237,70497,3805,"""VUSAPWT1B1""",0.0,"""1""","""1""","""1""",1.0,1.0,"""NBPF19"""
4623,11443,0,"""VUSAPWT1B2""",0.0,"""1""","""1""","""1""",null,null,"""KLHL12"""
6281,100016069,2639,"""VUSAPWT6B1""",0.0,"""6""","""1""","""1""",1.0,2.0,"""KCNJ15"""
56141,4870,215286,"""VUSAPWT1B2""",0.5625,"""1""","""1""","""1""",1.0,1.0,"""KRT31"""
…,…,…,…,…,…,…,…,…,…,…
54668,70497,3851,"""VUSAPWT1B1""",0.25,"""1""","""1""","""1""",1.0,1.0,"""KRT40"""
4636,5904,0,"""VUSAPWT1B2""",0.625,"""1""","""1""","""1""",null,null,"""TARBP2"""
11621,4062,201430,"""VUSAPWT6B1""",0.0,"""6""","""1""","""1""",1.0,1.0,"""MTIF3"""


In [74]:
# Append metadata and write out file
metadata = pl.read_csv(f"{meta_outputs}/slim_metadata.csv").rename({
    "orf_id": "db_orf_id",
    "mut_id": "db_mut_id"
})

normalized_scores = normalized_scores.join(metadata, on=["db_orf_id", "db_mut_id"], how="left")
normalized_scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
i64,i64,i64,str,f64,str,str,str,f64,f64,str,str,str,str,str,str,str,f64,f64,str
2860,7982,202790,"""VUSAPWT1B2""",1.0,"""1""","""1""","""1""",1.0,1.0,"""PRTFDC1""","""HPRT1""","""Asp80Val""","""239A>T""","""ENSG00000165704""","""CEGS2""","""1_Pathogenic""",null,1.0,"""HPRT1_Asp80Val"""
55237,70497,3805,"""VUSAPWT1B1""",0.0,"""1""","""1""","""1""",1.0,1.0,"""NBPF19""","""AGXT""","""Ala85Asp""","""254C>A""","""ENSG00000172482""","""Edgotyping3""","""1_Pathogenic""",null,null,"""AGXT_Ala85Asp"""
4623,11443,0,"""VUSAPWT1B2""",0.0,"""1""","""1""","""1""",null,null,"""KLHL12""","""CCBE1""",null,null,"""ENSG00000183287""",null,null,null,null,"""CCBE1"""
6281,100016069,2639,"""VUSAPWT6B1""",0.0,"""6""","""1""","""1""",1.0,2.0,"""KCNJ15""","""KCNJ2""","""Val295Ala""","""884T>C""","""ENSG00000123700""","""Edgotyping3""","""4_VUS""",null,2.0,"""KCNJ2_Val295Ala"""
56141,4870,215286,"""VUSAPWT1B2""",0.5625,"""1""","""1""","""1""",1.0,1.0,"""KRT31""","""KRT6A""","""Glu472Lys""","""1414G>A""","""ENSG00000205420""","""CEGS2""","""1_Pathogenic""",null,2.0,"""KRT6A_Glu472Lys"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
54668,70497,3851,"""VUSAPWT1B1""",0.25,"""1""","""1""","""1""",1.0,1.0,"""KRT40""","""AGXT""","""Ser221Pro""","""661T>C""","""ENSG00000172482""","""Edgotyping3""","""1_Pathogenic""",null,null,"""AGXT_Ser221Pro"""
4636,5904,0,"""VUSAPWT1B2""",0.625,"""1""","""1""","""1""",null,null,"""TARBP2""","""PRKRA""",null,null,"""ENSG00000180228""",null,null,null,null,"""PRKRA"""
11621,4062,201430,"""VUSAPWT6B1""",0.0,"""6""","""1""","""1""",1.0,1.0,"""MTIF3""","""PLP1""","""Cys193Tyr""","""578G>A""","""ENSG00000123560""","""CEGS2""","""1_Pathogenic""",0.0,1.0,"""PLP1_Cys193Tyr"""


In [75]:
normalized_scores.write_csv(f"{outputs_dir}/4_normalized_consensus_scores.csv")

In [77]:
normalized_scores_manual = normalized_scores

# Exploring qa/qc, dropped alleles

In [ ]:
def describe_dict(df_dict):
    for assay_id in df_dict.keys():
        print(f"\tAssay ID: {assay_id}")
        n_db_orfs = len(df_dict[assay_id].keys())
        only_wt_count = 0
        db_list_wt = []
        n_wt = 0
        n_alleles = 0
        db_list_wt_with_var_wt = []
        db_list_wt_with_only_var = []
        ad_list = []
        
        for db_orf_id, db_mut_d in df_dict[assay_id].items():
            # if db_orf_id == 53798:
            #     print('53798')
            #     print(db_mut_d)
            
            if 0 in db_mut_d.keys():
                n_wt += 1
                n_alleles += len(db_mut_d.keys()) - 1
            else:
                n_alleles += len(db_mut_d.keys())
                
            if len(db_mut_d.keys()) == 1 and 0 in db_mut_d.keys():
                only_wt_count += 1
                db_list_wt.append(db_orf_id)
                
            if len(db_mut_d.keys()) > 1 and 0 in db_mut_d.keys():
                db_list_wt_with_var_wt.append(db_orf_id)
                
            if 0 not in db_mut_d.keys():
                db_list_wt_with_only_var.append(db_orf_id)
                
            ad_set = set()
            for db_mut_id, ad_orf_id_set in db_mut_d.items():
                for ad_orf_id in ad_orf_id_set:
                    ad_set.add(ad_orf_id)
                    
            ad_list.extend(list(ad_set))
            
            
        print(f"\t\tNumber of DB ORFs: {n_db_orfs}")        
        print(f"\t\tNumber of DB alleles: {n_alleles}")
        print(f"\t\tNumber of DB ORFs with WT: {n_wt}")
        print(f"\t\tNumber of DB ORFs with only WT: {only_wt_count}, {(only_wt_count/n_db_orfs)*100:.2f}%")
        print(f"\t\tNumber of DB ORFs with WT and alleles: {len(db_list_wt_with_var_wt)}, {(len(db_list_wt_with_var_wt)/n_db_orfs)*100:.2f}%")
        print(f"\t\tNumber of DB ORFs with only alleles: {len(db_list_wt_with_only_var)}, {(len(db_list_wt_with_only_var)/n_db_orfs)*100:.2f}%")
        print(f"\t\tNumber of AD ORFs: {len(ad_list)}")

        print(f"\t\tDB ORFs with only WT: {db_list_wt}")
        print(f"\t\tDB ORFs with WT and alleles: {db_list_wt_with_var_wt}")
        print(f"\t\tDB ORFs with only alleles: {db_list_wt_with_only_var}")
        print()

In [ ]:
def compare_dicts(dict1, dict2):
    assay_array = ["VUSAPWT1B1", "VUSAPWT1B2", "VUSAPWT2B1", "VUSAPWT6B1"]
    for assay in assay_array:
        print(f"\tAssay: {assay}")

        lost_alleles = 0
        lost_wt = 0
        lost_db_orfs = 0
        lost_ad_orfs = 0

        total_alleles = 0
        total_wt = 0
        total_db_orfs = 0
        total_ad_orfs = 0

        # Compute totals from dict1
        for db_orf_id, db_mut_id_d in dict1[assay].items():
            total_db_orfs += 1
            if 0 in db_mut_id_d:
                total_wt += 1
                total_alleles += len(db_mut_id_d) - 1  # exclude WT
            else:
                total_alleles += len(db_mut_id_d)
                
            ad_set = set()
                
            ad_set = {ad for db_mut, ad_orf in db_mut_id_d.items() for ad in ad_orf}
            total_ad_orfs += len(ad_set)

        for db_orf_id, db_mut_id_d in dict2[assay].items():
            if db_orf_id in dict1[assay]:
                db_mut_id_d1 = dict1[assay][db_orf_id]
                db_mut_id_d2 = db_mut_id_d

                ad_set_1 = {ad for db_mut, ad_orf in db_mut_id_d1.items() for ad in ad_orf}
                ad_set_2 = {ad for db_mut, ad_orf in db_mut_id_d2.items() for ad in ad_orf}
                lost_ad_orfs += len(ad_set_1 - ad_set_2)

                lost_alleles += sum(1 for db_mut in db_mut_id_d1 if db_mut != 0 and db_mut not in db_mut_id_d2)

                if 0 in db_mut_id_d1 and 0 not in db_mut_id_d2:
                    lost_wt += 1

            else:
                lost_db_orfs += 1
                if 0 in db_mut_id_d:
                    lost_wt += 1
                    lost_alleles += len(db_mut_id_d) - 1
                else:
                    lost_alleles += len(db_mut_id_d)

                lost_ad_orfs += len({ad for db_mut, ad_orf in db_mut_id_d.items() for ad in ad_orf})

        for db_orf_id, db_mut_id_d in dict1[assay].items():
            if db_orf_id not in dict2[assay]:
                lost_db_orfs += 1
                if 0 in db_mut_id_d:
                    lost_wt += 1
                    lost_alleles += len(db_mut_id_d) - 1
                else:
                    lost_alleles += len(db_mut_id_d)
                lost_ad_orfs += len({ad for db_mut, ad_orf in db_mut_id_d.items() for ad in ad_orf})

        def pct(x, total):
            return f"{(x / total * 100):.1f}%" if total else "NA"

        print(f"\t\tLost DB ORFs: {lost_db_orfs} / {total_db_orfs} ({pct(lost_db_orfs, total_db_orfs)})")
        print(f"\t\tLost alleles: {lost_alleles} / {total_alleles} ({pct(lost_alleles, total_alleles)})")
        print(f"\t\tLost WT: {lost_wt} / {total_wt} ({pct(lost_wt, total_wt)})")
        print(f"\t\tLost AD ORFs: {lost_ad_orfs} / {total_ad_orfs} ({pct(lost_ad_orfs, total_ad_orfs)})")
        print()


In [ ]:
dicts = [dict_before_qc, dict_after_db_groups, dict_after_db_conditions ,dict_after_mating, dict_after_aa, dict_after_seq_1, dict_after_seq_2]

In [ ]:
print("Data filtering through QC/QA:")
print()
print()
print('Before QC:')
print()
describe_dict(dict_before_qc)
print()
print('Step 1: Remove all db groups with the same allele if there was an autoactivation mating failure')
print()
describe_dict(dict_after_db_groups)
print()
print('Differences before QC and step 1:')
print()
compare_dicts(dict_before_qc, dict_after_db_groups)
print()
print('________________________________________________________________________________')
print('Step 2: Remove all db conditions with the same allele if there was an autoactivation mating failure')
print()
describe_dict(dict_after_db_conditions)
print()
print()
print('Differences between step 1 and step 2:')
print()
compare_dicts(dict_after_db_groups, dict_after_db_conditions)
print()
print('________________________________________________________________________________')
print('Step 3: remove mating failures - condition 1 shows no or little yeast growth')
print()
describe_dict(dict_after_mating)
print()
print('Differences between step 2 and step 3:')
print()
compare_dicts(dict_after_db_conditions, dict_after_mating)
print()
print('________________________________________________________________________________')
print('Step 4: Remove autoactivation rows from the main scores df')
print()
describe_dict(dict_after_aa)
print()
print('Differences between step 3 and step 4:')
print()
compare_dicts(dict_after_mating, dict_after_aa)
print()
print('________________________________________________________________________________')
print('Step 5: Filter to only keep interactions where both ad and db are confirmed, before normalizing')
print()
describe_dict(dict_after_seq_1)
print()
print('Differences between step 4 and step 5:')
print()
compare_dicts(dict_after_aa, dict_after_seq_1)
print()
print('________________________________________________________________________________')
print('Step 6: Filter to only keep interactions where both ad and db are confirmed, after normalizing')
print()
describe_dict(dict_after_seq_2)
print()
print('Differences between step 5 and step 6:')
print()
compare_dicts(dict_after_seq_1, dict_after_seq_2)


Data filtering through QC/QA:


Before QC:

	Assay ID: VUSAPWT1B1
		Number of DB ORFs: 23
		Number of DB alleles: 589
		Number of DB ORFs with WT: 23
		Number of DB ORFs with only WT: 0, 0.00%
		Number of DB ORFs with WT and alleles: 23, 100.00%
		Number of DB ORFs with only alleles: 0, 0.00%
		Number of AD ORFs: 104
		DB ORFs with only WT: []
		DB ORFs with WT and alleles: [281, 2651, 2713, 3535, 4723, 4765, 7167, 7215, 7323, 7652, 11805, 12573, 13295, 52920, 54308, 54914, 55246, 56661, 70497, 71001, 71337, 71892, 100016069]
		DB ORFs with only alleles: []

	Assay ID: VUSAPWT1B2
		Number of DB ORFs: 251
		Number of DB alleles: 506
		Number of DB ORFs with WT: 251
		Number of DB ORFs with only WT: 107, 42.63%
		Number of DB ORFs with WT and alleles: 144, 57.37%
		Number of DB ORFs with only alleles: 0, 0.00%
		Number of AD ORFs: 910
		DB ORFs with only WT: [138, 253, 268, 747, 1155, 1284, 1485, 1601, 1884, 2012, 2055, 2380, 2430, 2442, 2448, 2613, 2816, 2818, 2866, 3125, 3192, 3292, 35

In [ ]:
compare_dicts(dict_before_qc, dict_after_mating)

	Assay: VUSAPWT1B1
		Lost DB ORFs: 0 / 23 (0.0%)
		Lost alleles: 8 / 589 (1.4%)
		Lost WT: 0 / 23 (0.0%)
		Lost AD ORFs: 0 / 104 (0.0%)

	Assay: VUSAPWT1B2
		Lost DB ORFs: 1 / 251 (0.4%)
		Lost alleles: 7 / 506 (1.4%)
		Lost WT: 1 / 251 (0.4%)
		Lost AD ORFs: 2 / 910 (0.2%)

	Assay: VUSAPWT2B1
		Lost DB ORFs: 4 / 64 (6.2%)
		Lost alleles: 11 / 275 (4.0%)
		Lost WT: 5 / 64 (7.8%)
		Lost AD ORFs: 26 / 256 (10.2%)

	Assay: VUSAPWT6B1
		Lost DB ORFs: 0 / 83 (0.0%)
		Lost alleles: 10 / 361 (2.8%)
		Lost WT: 0 / 83 (0.0%)
		Lost AD ORFs: 4 / 322 (1.2%)



In [ ]:
compare_dicts(dict_after_mating, dict_after_aa)

	Assay: VUSAPWT1B1
		Lost DB ORFs: 0 / 23 (0.0%)
		Lost alleles: 0 / 581 (0.0%)
		Lost WT: 0 / 23 (0.0%)
		Lost AD ORFs: 0 / 104 (0.0%)

	Assay: VUSAPWT1B2
		Lost DB ORFs: 0 / 250 (0.0%)
		Lost alleles: 10 / 499 (2.0%)
		Lost WT: 0 / 250 (0.0%)
		Lost AD ORFs: 0 / 908 (0.0%)

	Assay: VUSAPWT2B1
		Lost DB ORFs: 1 / 60 (1.7%)
		Lost alleles: 1 / 264 (0.4%)
		Lost WT: 1 / 59 (1.7%)
		Lost AD ORFs: 2 / 230 (0.9%)

	Assay: VUSAPWT6B1
		Lost DB ORFs: 0 / 83 (0.0%)
		Lost alleles: 0 / 351 (0.0%)
		Lost WT: 0 / 83 (0.0%)
		Lost AD ORFs: 0 / 318 (0.0%)



In [ ]:
compare_dicts(dict_after_aa, dict_after_seq_1)

	Assay: VUSAPWT1B1
		Lost DB ORFs: 2 / 23 (8.7%)
		Lost alleles: 311 / 581 (53.5%)
		Lost WT: 2 / 23 (8.7%)
		Lost AD ORFs: 25 / 104 (24.0%)

	Assay: VUSAPWT1B2
		Lost DB ORFs: 33 / 250 (13.2%)
		Lost alleles: 181 / 489 (37.0%)
		Lost WT: 36 / 250 (14.4%)
		Lost AD ORFs: 167 / 908 (18.4%)

	Assay: VUSAPWT2B1
		Lost DB ORFs: 7 / 59 (11.9%)
		Lost alleles: 106 / 263 (40.3%)
		Lost WT: 7 / 58 (12.1%)
		Lost AD ORFs: 49 / 228 (21.5%)

	Assay: VUSAPWT6B1
		Lost DB ORFs: 6 / 83 (7.2%)
		Lost alleles: 155 / 351 (44.2%)
		Lost WT: 7 / 83 (8.4%)
		Lost AD ORFs: 20 / 318 (6.3%)



In [ ]:
describe_dict(dict_before_qc)

	Assay ID: VUSAPWT1B1
		Number of DB ORFs: 23
		Number of DB alleles: 589
		Number of DB ORFs with WT: 23
		Number of DB ORFs with only WT: 0, 0.00%
		Number of DB ORFs with WT and alleles: 23, 100.00%
		Number of DB ORFs with only alleles: 0, 0.00%
		Number of AD ORFs: 104
		DB ORFs with only WT: []
		DB ORFs with WT and alleles: [281, 2651, 2713, 3535, 4723, 4765, 7167, 7215, 7323, 7652, 11805, 12573, 13295, 52920, 54308, 54914, 55246, 56661, 70497, 71001, 71337, 71892, 100016069]
		DB ORFs with only alleles: []

	Assay ID: VUSAPWT1B2
		Number of DB ORFs: 251
		Number of DB alleles: 506
		Number of DB ORFs with WT: 251
		Number of DB ORFs with only WT: 107, 42.63%
		Number of DB ORFs with WT and alleles: 144, 57.37%
		Number of DB ORFs with only alleles: 0, 0.00%
		Number of AD ORFs: 910
		DB ORFs with only WT: [138, 253, 268, 747, 1155, 1284, 1485, 1601, 1884, 2012, 2055, 2380, 2430, 2442, 2448, 2613, 2816, 2818, 2866, 3125, 3192, 3292, 3594, 3643, 3668, 3670, 3693, 3699, 3844, 3878

In [ ]:
describe_dict(dict_after_mating)

	Assay ID: VUSAPWT1B1
		Number of DB ORFs: 23
		Number of DB alleles: 581
		Number of DB ORFs with WT: 23
		Number of DB ORFs with only WT: 0, 0.00%
		Number of DB ORFs with WT and alleles: 23, 100.00%
		Number of DB ORFs with only alleles: 0, 0.00%
		Number of AD ORFs: 104
		DB ORFs with only WT: []
		DB ORFs with WT and alleles: [281, 2651, 2713, 3535, 4723, 4765, 7167, 7215, 7323, 7652, 11805, 12573, 13295, 52920, 54308, 54914, 55246, 56661, 70497, 71001, 71337, 71892, 100016069]
		DB ORFs with only alleles: []

	Assay ID: VUSAPWT1B2
		Number of DB ORFs: 250
		Number of DB alleles: 499
		Number of DB ORFs with WT: 250
		Number of DB ORFs with only WT: 107, 42.80%
		Number of DB ORFs with WT and alleles: 143, 57.20%
		Number of DB ORFs with only alleles: 0, 0.00%
		Number of AD ORFs: 908
		DB ORFs with only WT: [138, 253, 268, 747, 1155, 1284, 1485, 1601, 1884, 2012, 2055, 2380, 2430, 2442, 2448, 2613, 2816, 2818, 2866, 3125, 3192, 3292, 3594, 3643, 3668, 3670, 3693, 3699, 3844, 3878

In [ ]:
describe_dict(dict_after_aa)

	Assay ID: VUSAPWT1B1
		Number of DB ORFs: 23
		Number of DB alleles: 581
		Number of DB ORFs with WT: 23
		Number of DB ORFs with only WT: 0, 0.00%
		Number of DB ORFs with WT and alleles: 23, 100.00%
		Number of DB ORFs with only alleles: 0, 0.00%
		Number of AD ORFs: 104
		DB ORFs with only WT: []
		DB ORFs with WT and alleles: [281, 2651, 2713, 3535, 4723, 4765, 7167, 7215, 7323, 7652, 11805, 12573, 13295, 52920, 54308, 54914, 55246, 56661, 70497, 71001, 71337, 71892, 100016069]
		DB ORFs with only alleles: []

	Assay ID: VUSAPWT1B2
		Number of DB ORFs: 250
		Number of DB alleles: 489
		Number of DB ORFs with WT: 250
		Number of DB ORFs with only WT: 108, 43.20%
		Number of DB ORFs with WT and alleles: 142, 56.80%
		Number of DB ORFs with only alleles: 0, 0.00%
		Number of AD ORFs: 908
		DB ORFs with only WT: [138, 253, 268, 747, 1155, 1284, 1485, 1601, 1884, 2012, 2055, 2380, 2430, 2442, 2448, 2613, 2816, 2818, 2866, 3125, 3192, 3292, 3594, 3643, 3668, 3670, 3693, 3699, 3844, 3878

# Automated

In [78]:
inputs_dir = "../1_inputs/sqY2H"
outputs_dir = "../3_outputs/sqY2H"
meta_inputs = "../../../1_allele_collection/1_inputs"
meta_outputs = "../../../1_allele_collection/3_outputs"

scoring_conditions = ['LWH1', 'LWH10', 'LWH25', 'LWA']
max_score = 4
aa_id = 0
rrs_id = 999999.0

In [79]:
# get media control thresholds
media_score_thresholds = pd.read_csv(f"{inputs_dir}/media_control_score_thresholds.csv")
media_score_thresholds.rename(columns={'control_96': 'scoring_pos'}, inplace=True)

In [80]:
media_score_thresholds

,condition_name,scoring_pos,control_384,lower_bound,upper_bound
0,LW,A01,A01_A02_B01_B02,4,4
1,LW,B01,C01_C02_D01_D02,4,4
2,LW,C01,E01_E02_F01_F02,4,4
3,LW,D01,G01_G02_H01_H02,4,4
4,LW,E01,I01_I02_J01_J02,4,4
5,LW,F01,K01_K02_L01_L02,4,4
6,LWA,A01,A01_A02_B01_B02,0,0
7,LWA,B01,C01_C02_D01_D02,0,2
8,LWA,C01,E01_E02_F01_F02,0,2
9,LWA,D01,G01_G02_H01_H02,3,4


In [81]:
scores = pd.read_csv(f"{outputs_dir}/2_automated_raw_consensus_scores.csv") 

# Split into media controls
media_ctrls = scores[scores['media_control']]
scores = scores[~scores['media_control']]

# Check media controls
media_ctrls = media_ctrls[['db_orf_id', 'condition_name', 'retest_batch', 'consensus_score']]
media_ctrls[['scoring_pos', 'retest_pla']] = media_ctrls['db_orf_id'].str.split('_', n=1, expand=True)
media_ctrls = media_ctrls.merge(media_score_thresholds, on=['scoring_pos', 'condition_name'])
media_ctrls['score_in_bounds'] = media_ctrls['consensus_score'].between(media_ctrls['lower_bound'], media_ctrls['upper_bound'], inclusive='both')

# replace failure due to variable scoring (99999) with NaN to group with other technical failures
scores['consensus_score'] = scores['consensus_score'].replace(99999, np.nan)

# Fix data types before converting to polars
scores['db_orf_id'] = pd.to_numeric(scores['db_orf_id'], errors='coerce')
scores['db_mut_id'] = pd.to_numeric(scores['db_mut_id'], errors='coerce')
scores['ad_orf_id'] = pd.to_numeric(scores['ad_orf_id'], errors='coerce')
scores['ad_mut_id'] = pd.to_numeric(scores['ad_mut_id'], errors='coerce')
scores['consensus_score'] = pd.to_numeric(scores['consensus_score'], errors='coerce')

# add IDs for groups used for filtering
scores = pl.DataFrame(scores)

scores = scores.with_columns(
    pl.col("retest_batch").str.extract(r"VUSAPWT(\d)B\d+", 1).alias("assay_id")
)

scores = scores.with_columns(
    pl.concat_str([
            pl.col('db_orf_id').cast(pl.String),
            pl.col('db_mut_id').cast(pl.String)
    ], separator="_").alias("db_group"),
    pl.concat_str([
            pl.col('db_orf_id').cast(pl.String),
            pl.col('db_mut_id').cast(pl.String),
            pl.col('condition_name').cast(pl.String),
            pl.col('retest_batch').cast(pl.String)
    ], separator="_").alias("db_condition_group"),
    pl.concat_str([
            pl.col('ad_orf_id').cast(pl.String),
            pl.col('db_orf_id').cast(pl.String),
            pl.col('db_mut_id').cast(pl.String),
            pl.col('retest_batch').cast(pl.String)
    ], separator="_").alias("ppi_group")
)

scores

scoring_pla,scoring_pos,large_plate_name,retest_batch,condition,condition_name,assay_id,media_control,ad_orf_id,ad_mut_id,db_orf_id,db_mut_id,db_source_plate,db_source_well,ad_source_plate,ad_source_well,manual_score_growth,per_well_integrated_intensity_gray,consensus_score,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0,db_group,db_condition_group,ppi_group
str,str,str,str,str,str,str,bool,f64,f64,f64,f64,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,f64,f64,str,str,str,str
"""VUSAPWT1B1r01_001""","""A03""","""VUSAPWT1B1r01_001_002_003_004_…","""VUSAPWT1B1""","""r07""","""LW""","""1""",false,0.0,null,2651.0,0.0,"""DisWTGDEh02""","""G09""","""CEGSint_Y8A1_2_6""","""A01""",4.0,455.65623,494.701983,"""empty_ad""","""CLN6""",null,null,"""ENSG00000128973""",null,null,null,null,"""CLN6""","""2651.0_0.0""","""2651.0_0.0_LW_VUSAPWT1B1""","""0.0_2651.0_0.0_VUSAPWT1B1"""
"""VUSAPWT1B1r01_001""","""A05""","""VUSAPWT1B1r01_001_002_003_004_…","""VUSAPWT1B1""","""r07""","""LW""","""1""",false,0.0,null,7215.0,0.0,"""DisWTGDEh03""","""E01""","""CEGSint_Y8A1_2_6""","""A01""",4.0,505.925513,512.738259,"""empty_ad""","""NR0B1""",null,null,"""ENSG00000169297""",null,null,null,null,"""NR0B1""","""7215.0_0.0""","""7215.0_0.0_LW_VUSAPWT1B1""","""0.0_7215.0_0.0_VUSAPWT1B1"""
"""VUSAPWT1B1r01_001""","""A07""","""VUSAPWT1B1r01_001_002_003_004_…","""VUSAPWT1B1""","""r07""","""LW""","""1""",false,279.0,null,11451.0,9.99999e5,"""RRS_VUS1_DB_1""","""B01""","""RRS_VUS1_AD_1""","""B01""",4.0,498.666689,532.80656,"""CALCOCO2""",null,null,null,null,null,null,null,null,null,"""11451.0_999999.0""","""11451.0_999999.0_LW_VUSAPWT1B1""","""279.0_11451.0_999999.0_VUSAPWT…"
"""VUSAPWT1B1r01_001""","""A09""","""VUSAPWT1B1r01_001_002_003_004_…","""VUSAPWT1B1""","""r07""","""LW""","""1""",false,0.0,null,1.00016069e8,0.0,"""DisWTGDEh03""","""G03""","""CEGSint_Y8A1_2_6""","""A01""",4.0,496.091526,528.079763,"""empty_ad""","""KCNJ2""",null,null,"""ENSG00000123700""",null,null,null,null,"""KCNJ2""","""100016069.0_0.0""","""100016069.0_0.0_LW_VUSAPWT1B1""","""0.0_100016069.0_0.0_VUSAPWT1B1"""
"""VUSAPWT1B1r01_001""","""A11""","""VUSAPWT1B1r01_001_002_003_004_…","""VUSAPWT1B1""","""r07""","""LW""","""1""",false,15066.0,null,2713.0,0.0,"""DisWTGDEh03""","""A04""","""CEGSint_Y8A_3""","""F07""",4.0,557.534667,569.41408,"""CXorf57""","""MLH1""",null,null,"""ENSG00000076242""",null,null,null,null,"""MLH1""","""2713.0_0.0""","""2713.0_0.0_LW_VUSAPWT1B1""","""15066.0_2713.0_0.0_VUSAPWT1B1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""VUSAPWT1B1r06_021""","""A01""","""VUSAPWT1B1r06_019_020_021_022_…","""VUSAPWT1B1""","""r12""","""LWAH1""","""1""",false,0.0,null,1.00016069e8,2639.0,"""VUSMutY8Dh010""","""E06""","""CEGSint_Y8A1_2_6""","""B01""",0.0,281.69543,null,"""empty_ad""","""KCNJ2""","""Val295Ala""","""884T>C""","""ENSG00000123700""","""Edgotyping3""","""4_VUS""",null,2.0,"""KCNJ2_Val295Ala""","""100016069.0_2639.0""","""100016069.0_2639.0_LWAH1_VUSAP…","""0.0_100016069.0_2639.0_VUSAPWT…"
"""VUSAPWT1B1r06_021""","""C01""","""VUSAPWT1B1r06_019_020_021_022_…","""VUSAPWT1B1""","""r12""","""LWAH1""","""1""",false,1.00016069e8,null,1.00016069e8,2639.0,"""VUSMutY8Dh010""","""E06""","""CEGSint_Y8A_7""","""B04""",0.0,611.602624,null,"""KCNJ2""","""KCNJ2""","""Val295Ala""","""884T>C""","""ENSG00000123700""","""Edgotyping3""","""4_VUS""",null,2.0,"""KCNJ2_Val295Ala""","""100016069.0_2639.0""","""100016069.0_2639.0_LWAH1_VUSAP…","""100016069.0_100016069.0_2639.0…"
"""VUSAPWT1B1r06_034""","""E01""","""VUSAPWT1B1r06_031_032_033_034_…","""VUSAPWT1B1""","""r12""","""LWAH1""","""1""",false,4621.0,null,4723.0,201746.0,"""VUSMutY8Dh019""","""C06""","""CEGSint_Y8A_2""","""G09""",3.0,322.449677,null,"""KXD1""","""TPM1""","""Ala239Thr""","""715G>A""","""ENSG00000140416""","""CEGS2""","""4_VUS""",null,2.0,"""TPM1_Ala239Thr""","""4723.0_201746.0""","""4723.0_201746.0_LWAH1_VUSAPWT1…","""4621.0_4723.0_2

In [82]:
# Remove all db groups with the same allele if there was an autoactivation mating failure
aa_db_group_mating_failures = (scores.filter(pl.col("ad_orf_id") == aa_id)
                              .filter(pl.col("condition_name") == "LW")
                              .filter(pl.col("consensus_score") <= 300)
                              .select("db_group")
                              .to_series().unique().to_list())
scores = scores.filter(~pl.col("db_group").is_in(aa_db_group_mating_failures))


# Remove all db-condition groups with the same allele if there was an autoactivation mating failure
aa_db_group_tech_failures = (scores.filter(pl.col("ad_orf_id") == aa_id)
                                   .filter(pl.col("consensus_score").is_nan())
                                   .select("db_condition_group")
                                   .to_series().unique().to_list())
scores = scores.filter(~pl.col("db_condition_group").is_in(aa_db_group_tech_failures))


# remove mating failures - condition 1 shows no or little yeast growth
mating_failures = (scores.filter(pl.col("condition_name") == "LW")
                         .filter(pl.col("consensus_score") <= 300)
                         .select("ppi_group").to_series().unique().to_list())
scores = scores.filter(~pl.col("ppi_group").is_in(mating_failures))


# Adjust consensus score based on autoactivation score
aa_scores = scores.filter(pl.col("ad_orf_id") == aa_id).select([
    "db_condition_group", "consensus_score"
]).rename({"consensus_score": "aa_score"})

# Remove autoactivation rows from the main scores df
# Add the aa_score to the dataframe, merged on the condition-specific db allele
scores = scores.filter(pl.col("ad_orf_id") != aa_id).join(aa_scores, on="db_condition_group")

scores = scores.with_columns(
    (pl.col("consensus_score") - pl.col("aa_score")).alias("adjusted_score")
).filter(
   ~((pl.col("adjusted_score") <= 0) & (pl.col("aa_score") > 0))
)

scores

scoring_pla,scoring_pos,large_plate_name,retest_batch,condition,condition_name,assay_id,media_control,ad_orf_id,ad_mut_id,db_orf_id,db_mut_id,db_source_plate,db_source_well,ad_source_plate,ad_source_well,manual_score_growth,per_well_integrated_intensity_gray,consensus_score,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0,db_group,db_condition_group,ppi_group,aa_score,adjusted_score
str,str,str,str,str,str,str,bool,f64,f64,f64,f64,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,f64,f64,str,str,str,str,f64,f64
"""VUSAPWT1B1r01_001""","""A11""","""VUSAPWT1B1r01_001_002_003_004_…","""VUSAPWT1B1""","""r07""","""LW""","""1""",false,15066.0,null,2713.0,0.0,"""DisWTGDEh03""","""A04""","""CEGSint_Y8A_3""","""F07""",4.0,557.534667,569.41408,"""CXorf57""","""MLH1""",null,null,"""ENSG00000076242""",null,null,null,null,"""MLH1""","""2713.0_0.0""","""2713.0_0.0_LW_VUSAPWT1B1""","""15066.0_2713.0_0.0_VUSAPWT1B1""",499.708842,69.705237
"""VUSAPWT1B1r01_001""","""A15""","""VUSAPWT1B1r01_001_002_003_004_…","""VUSAPWT1B1""","""r07""","""LW""","""1""",false,71704.0,null,11805.0,0.0,"""DisWTGDEh03""","""C04""","""CEGSint_Y8A_1""","""C01""",4.0,528.001331,558.350679,"""SMURF2""","""SMAD3""",null,null,"""ENSG00000166949""",null,null,null,null,"""SMAD3""","""11805.0_0.0""","""11805.0_0.0_LW_VUSAPWT1B1""","""71704.0_11805.0_0.0_VUSAPWT1B1""",509.083347,49.267332
"""VUSAPWT1B1r01_001""","""A17""","""VUSAPWT1B1r01_001_002_003_004_…","""VUSAPWT1B1""","""r07""","""LW""","""1""",false,13662.0,null,70497.0,0.0,"""DisWTGDEh03""","""D04""","""CEGSint_Y8A_5""","""H07""",4.0,585.556892,586.988591,"""CYSRT1""","""AGXT""",null,null,"""ENSG00000172482""",null,null,null,null,"""AGXT""","""70497.0_0.0""","""70497.0_0.0_LW_VUSAPWT1B1""","""13662.0_70497.0_0.0_VUSAPWT1B1""",515.308517,71.680074
"""VUSAPWT1B1r01_001""","""A19""","""VUSAPWT1B1r01_001_002_003_004_…","""VUSAPWT1B1""","""r07""","""LW""","""1""",false,10460.0,null,12573.0,0.0,"""DisWTGDEh03""","""E04""","""CEGSint_Y8A_1""","""A09""",4.0,577.207873,572.116368,"""GOLGA6L9""","""DES""",null,null,"""ENSG00000175084""",null,null,null,null,"""DES""","""12573.0_0.0""","""12573.0_0.0_LW_VUSAPWT1B1""","""10460.0_12573.0_0.0_VUSAPWT1B1""",525.32257,46.793798
"""VUSAPWT1B1r01_001""","""A21""","""VUSAPWT1B1r01_001_002_003_004_…","""VUSAPWT1B1""","""r07""","""LW""","""1""",false,5894.0,null,12573.0,0.0,"""DisWTGDEh03""","""E04""","""CEGSint_Y8A_6""","""E09""",4.0,574.316367,587.623557,"""TFIP11""","""DES""",null,null,"""ENSG00000175084""",null,null,null,null,"""DES""","""12573.0_0.0""","""12573.0_0.0_LW_VUSAPWT1B1""","""5894.0_12573.0_0.0_VUSAPWT1B1""",525.32257,62.300987
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""VUSAPWT1B1r04_035""","""K13""","""VUSAPWT1B1r04_031_032_033_034_…","""VUSAPWT1B1""","""r10""","""LWH10""","""1""",false,2920.0,null,12573.0,203769.0,"""VUSMutY8Dh022""","""A02""","""CEGSint_Y8A_4""","""E01""",2.0,394.631384,null,"""GFAP""","""DES""","""Leu274Pro""","""821T>C""","""ENSG00000175084""","""CEGS2""","""1_Pathogenic""",null,1.0,"""DES_Leu274Pro""","""12573.0_203769.0""","""12573.0_203769.0_LWH10_VUSAPWT…","""2920.0_12573.0_203769.0_VUSAPW…",0.0,null
"""VUSAPWT1B1r06_022""","""C11""","""VUSAPWT1B1r06_019_020_021_022_…","""VUSAPWT1B1""","""r12""","""LWAH1""","""1""",false,70264.0,null,2713.0,470.0,"""VUSMutY8Dh011""","""D08""","""CEGSint_Y8A_3""","""G02""",0.0,0.0,null,"""PPP1R13B""","""MLH1""","""Asn38Lys""","""114C>G""","""ENSG00000076242""","""Edgotyping3""","""1_Pathogenic""",null,3.0,"""MLH1_Asn38Lys""","""2713.0_470.0""","""2713.0_470.0_LWAH1_VUSAPWT1B1""","""70264.0_2713.0_470.0_VUSAPWT1B…",0.0,null
"""VUSAPWT1B1r06_034""","""E01""","""VUSAPWT1B1r06_031_032_033_034_…","""VUSAPWT1B1""","""r12""","""LWAH1""","""1""",false,4621.0,null,4723.0,201746.0,"""VUSMutY8Dh019""","""C06""","""CEGSint_Y8A_2""","""G09""",3.0,322.449677,null,"""KXD1""","""TPM1""","""Ala239Thr""","""715G>A""","""ENSG0000014041

In [83]:
print(aa_db_group_mating_failures)
print(len(aa_db_group_mating_failures))
print(aa_db_group_tech_failures)
print(len(aa_db_group_tech_failures))
print(mating_failures)
print(len(mating_failures))

['2920.0_214415.0', '100016069.0_2629.0', '71592.0_0.0', '100016069.0_2650.0', '3756.0_999999.0', '52920.0_2668.0', '71077.0_220310.0', '100016069.0_2604.0', '100016069.0_2637.0', '100016069.0_2638.0', '12705.0_999999.0', '636.0_999999.0', '3668.0_0.0', '11443.0_999999.0', '281.0_29.0', '100016069.0_2620.0', '8151.0_999999.0', '281.0_12.0', '100016069.0_2624.0', '55222.0_999999.0']
20
[]
0
['8165.0_4062.0_201423.0_VUSAPWT6B1', '71567.0_4862.0_0.0_VUSAPWT2B1', '11539.0_2713.0_472.0_VUSAPWT6B1', '3068.0_71077.0_220313.0_VUSAPWT2B1', '730.0_52920.0_2675.0_VUSAPWT6B1', '11130.0_4062.0_201432.0_VUSAPWT6B1', '100015467.0_5066.0_201832.0_VUSAPWT6B1', '100069030.0_7038.0_216141.0_VUSAPWT2B1', '6281.0_100016069.0_2618.0_VUSAPWT6B1', '9501.0_5723.0_201966.0_VUSAPWT2B1', '3907.0_5066.0_201830.0_VUSAPWT1B2', '56547.0_52920.0_218636.0_VUSAPWT2B1', '13219.0_54230.0_218818.0_VUSAPWT6B1', '71622.0_9634.0_0.0_VUSAPWT6B1', '56547.0_52920.0_2681.0_VUSAPWT2B1', '2077.0_53030.0_218690.0_VUSAPWT2B1', '439.0

In [84]:
# Keep only scores that aren't RRS pairs
scores = scores.filter(pl.col('db_mut_id') != rrs_id)

scores = scores.with_columns(
    pl.col("retest_batch").str.extract(r"VUSAPWT(\d)B\d+", 1).alias("assay_id")
)

scores

scoring_pla,scoring_pos,large_plate_name,retest_batch,condition,condition_name,assay_id,media_control,ad_orf_id,ad_mut_id,db_orf_id,db_mut_id,db_source_plate,db_source_well,ad_source_plate,ad_source_well,manual_score_growth,per_well_integrated_intensity_gray,consensus_score,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0,db_group,db_condition_group,ppi_group,aa_score,adjusted_score
str,str,str,str,str,str,str,bool,f64,f64,f64,f64,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,f64,f64,str,str,str,str,f64,f64
"""VUSAPWT1B1r01_001""","""A11""","""VUSAPWT1B1r01_001_002_003_004_…","""VUSAPWT1B1""","""r07""","""LW""","""1""",false,15066.0,null,2713.0,0.0,"""DisWTGDEh03""","""A04""","""CEGSint_Y8A_3""","""F07""",4.0,557.534667,569.41408,"""CXorf57""","""MLH1""",null,null,"""ENSG00000076242""",null,null,null,null,"""MLH1""","""2713.0_0.0""","""2713.0_0.0_LW_VUSAPWT1B1""","""15066.0_2713.0_0.0_VUSAPWT1B1""",499.708842,69.705237
"""VUSAPWT1B1r01_001""","""A15""","""VUSAPWT1B1r01_001_002_003_004_…","""VUSAPWT1B1""","""r07""","""LW""","""1""",false,71704.0,null,11805.0,0.0,"""DisWTGDEh03""","""C04""","""CEGSint_Y8A_1""","""C01""",4.0,528.001331,558.350679,"""SMURF2""","""SMAD3""",null,null,"""ENSG00000166949""",null,null,null,null,"""SMAD3""","""11805.0_0.0""","""11805.0_0.0_LW_VUSAPWT1B1""","""71704.0_11805.0_0.0_VUSAPWT1B1""",509.083347,49.267332
"""VUSAPWT1B1r01_001""","""A17""","""VUSAPWT1B1r01_001_002_003_004_…","""VUSAPWT1B1""","""r07""","""LW""","""1""",false,13662.0,null,70497.0,0.0,"""DisWTGDEh03""","""D04""","""CEGSint_Y8A_5""","""H07""",4.0,585.556892,586.988591,"""CYSRT1""","""AGXT""",null,null,"""ENSG00000172482""",null,null,null,null,"""AGXT""","""70497.0_0.0""","""70497.0_0.0_LW_VUSAPWT1B1""","""13662.0_70497.0_0.0_VUSAPWT1B1""",515.308517,71.680074
"""VUSAPWT1B1r01_001""","""A19""","""VUSAPWT1B1r01_001_002_003_004_…","""VUSAPWT1B1""","""r07""","""LW""","""1""",false,10460.0,null,12573.0,0.0,"""DisWTGDEh03""","""E04""","""CEGSint_Y8A_1""","""A09""",4.0,577.207873,572.116368,"""GOLGA6L9""","""DES""",null,null,"""ENSG00000175084""",null,null,null,null,"""DES""","""12573.0_0.0""","""12573.0_0.0_LW_VUSAPWT1B1""","""10460.0_12573.0_0.0_VUSAPWT1B1""",525.32257,46.793798
"""VUSAPWT1B1r01_001""","""A21""","""VUSAPWT1B1r01_001_002_003_004_…","""VUSAPWT1B1""","""r07""","""LW""","""1""",false,5894.0,null,12573.0,0.0,"""DisWTGDEh03""","""E04""","""CEGSint_Y8A_6""","""E09""",4.0,574.316367,587.623557,"""TFIP11""","""DES""",null,null,"""ENSG00000175084""",null,null,null,null,"""DES""","""12573.0_0.0""","""12573.0_0.0_LW_VUSAPWT1B1""","""5894.0_12573.0_0.0_VUSAPWT1B1""",525.32257,62.300987
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""VUSAPWT1B1r04_035""","""K13""","""VUSAPWT1B1r04_031_032_033_034_…","""VUSAPWT1B1""","""r10""","""LWH10""","""1""",false,2920.0,null,12573.0,203769.0,"""VUSMutY8Dh022""","""A02""","""CEGSint_Y8A_4""","""E01""",2.0,394.631384,null,"""GFAP""","""DES""","""Leu274Pro""","""821T>C""","""ENSG00000175084""","""CEGS2""","""1_Pathogenic""",null,1.0,"""DES_Leu274Pro""","""12573.0_203769.0""","""12573.0_203769.0_LWH10_VUSAPWT…","""2920.0_12573.0_203769.0_VUSAPW…",0.0,null
"""VUSAPWT1B1r06_022""","""C11""","""VUSAPWT1B1r06_019_020_021_022_…","""VUSAPWT1B1""","""r12""","""LWAH1""","""1""",false,70264.0,null,2713.0,470.0,"""VUSMutY8Dh011""","""D08""","""CEGSint_Y8A_3""","""G02""",0.0,0.0,null,"""PPP1R13B""","""MLH1""","""Asn38Lys""","""114C>G""","""ENSG00000076242""","""Edgotyping3""","""1_Pathogenic""",null,3.0,"""MLH1_Asn38Lys""","""2713.0_470.0""","""2713.0_470.0_LWAH1_VUSAPWT1B1""","""70264.0_2713.0_470.0_VUSAPWT1B…",0.0,null
"""VUSAPWT1B1r06_034""","""E01""","""VUSAPWT1B1r06_031_032_033_034_…","""VUSAPWT1B1""","""r12""","""LWAH1""","""1""",false,4621.0,null,4723.0,201746.0,"""VUSMutY8Dh019""","""C06""","""CEGSint_Y8A_2""","""G09""",3.0,322.449677,null,"""KXD1""","""TPM1""","""Ala239Thr""","""715G>A""","""ENSG0000014041

In [85]:
# read in sequence confirmation files
ad_seq = pl.read_csv(f"{inputs_dir}/swimseq_confirmation/vus_ad_interactors.csv").rename({
    "swim_seq_validated": "seq_ad"
}).select([
    "ad_orf_id", "assay_id", "seq_ad"
])

db_var_seq = pl.read_csv(f"{inputs_dir}/swimseq_confirmation/vus_db_alleles.csv").rename({
    "orf_id": "db_orf_id",
    "mut_id": "db_mut_id",
    "swim_seq_validated": "seq_db"
}).select(["db_orf_id", "db_mut_id", "seq_db"])

db_ref_seq = pl.read_csv(f"{inputs_dir}/swimseq_confirmation/wt_disease_nodes_summary.csv").rename({
    "disease_orf_id": "db_orf_id",
    "swim_seq_validated": "seq_db"
}).select(["db_orf_id", "seq_db"]).with_columns(
    pl.lit(0).cast(pl.Int64).alias("db_mut_id")
).select(["db_orf_id", "db_mut_id", "seq_db"])

db_seq = pl.concat([db_var_seq, db_ref_seq])
# print(db_seq)

## Add an extra QC step to check the sequence confirmation results
seq_confirm_res_df = pd.read_table(f"{meta_inputs}/seq_confirmation_results/VarChampSeqConfirmationResult.tsv")
# with pd.option_context('display.max_columns', None):
#     display(seq_confirm_res_df)

db_seq = pl.DataFrame(
    pd.merge(db_seq.to_pandas(), 
            seq_confirm_res_df[
                ["orf_id_wt","mutation_id_old","db_sequenced","db_sequence_confirmation_class"] ##"symbol","aa_change","nt_change","ccsb_mutation_id"
            ].rename({"orf_id_wt": "db_orf_id",
                      "mutation_id_old": "db_mut_id"}, axis=1), 
        on=["db_orf_id","db_mut_id"],
        how="left"
    )
)
print(db_seq)
# print(db_seq["db_sequenced"].unique())
# print(db_seq.filter(pl.col("db_sequenced")==0))

shape: (2_887, 5)
┌───────────┬───────────┬────────┬──────────────┬────────────────────────────────┐
│ db_orf_id ┆ db_mut_id ┆ seq_db ┆ db_sequenced ┆ db_sequence_confirmation_class │
│ ---       ┆ ---       ┆ ---    ┆ ---          ┆ ---                            │
│ i64       ┆ i64       ┆ str    ┆ f64          ┆ f64                            │
╞═══════════╪═══════════╪════════╪══════════════╪════════════════════════════════╡
│ 281       ┆ 727       ┆ NULL   ┆ 1.0          ┆ 1.0                            │
│ 281       ┆ 739       ┆ 1      ┆ 1.0          ┆ 1.0                            │
│ 281       ┆ 13        ┆ 1      ┆ 1.0          ┆ 1.0                            │
│ 281       ┆ 744       ┆ 1      ┆ 1.0          ┆ 1.0                            │
│ 281       ┆ 763       ┆ 1      ┆ 1.0          ┆ 1.0                            │
│ …         ┆ …         ┆ …      ┆ …            ┆ …                              │
│ 8514      ┆ 0         ┆ 1      ┆ null         ┆ null               

In [86]:
# cast all to same format
scores = scores.with_columns(
    pl.col("ad_orf_id").cast(pl.Float64).cast(pl.Int64).alias("ad_orf_id"),
    pl.col("db_orf_id").cast(pl.Float64).cast(pl.Int64).alias("db_orf_id"),
    pl.col("db_mut_id").cast(pl.Float64).cast(pl.Int64).alias("db_mut_id"),
    pl.col("assay_id").cast(pl.String).alias("assay_id"),
)

db_seq = db_seq.with_columns(
    pl.col("db_orf_id").cast(pl.Float64).cast(pl.Int64).alias("db_orf_id"),
    pl.col("db_mut_id").cast(pl.Float64).cast(pl.Int64).alias("db_mut_id"),
    pl.col("seq_db").cast(pl.String).alias("seq_db")
)

ad_seq = ad_seq.with_columns(
    pl.col("ad_orf_id").cast(pl.Float64).cast(pl.Int64).alias("ad_orf_id"),
    pl.col("seq_ad").cast(pl.String).alias("seq_ad"),
    pl.col("assay_id").cast(pl.String).alias("assay_id"),
)

scores = scores.join(ad_seq, on=["ad_orf_id", "assay_id"], how="left")
scores = scores.join(db_seq, on=["db_orf_id", "db_mut_id"], how="left")

# print(scores.filter(pl.col("db_mut_id")==0))

## Filter to only keep interactions where both ad and db are confirmed
## And also confirmed by the sequence confirmation results
valid_db_sequence_confirmation_class = [1., 2., 7., 99.]
scores = scores.filter(
    (pl.col("seq_ad") == "1") & (pl.col("seq_db") == "1") & (((pl.col("db_sequence_confirmation_class").is_in(valid_db_sequence_confirmation_class)) & (pl.col("db_sequenced").is_not_null())) | (pl.col("db_mut_id")==0))
)

## After sequence confirmation, we lost 25521 - 21895 = 3626 interactions
# print(scores.shape)
print(scores)

shape: (17_884, 38)
┌────────────┬────────────┬────────────┬────────────┬───┬────────┬────────┬────────────┬───────────┐
│ scoring_pl ┆ scoring_po ┆ large_plat ┆ retest_bat ┆ … ┆ seq_ad ┆ seq_db ┆ db_sequenc ┆ db_sequen │
│ a          ┆ s          ┆ e_name     ┆ ch         ┆   ┆ ---    ┆ ---    ┆ ed         ┆ ce_confir │
│ ---        ┆ ---        ┆ ---        ┆ ---        ┆   ┆ str    ┆ str    ┆ ---        ┆ mation_cl │
│ str        ┆ str        ┆ str        ┆ str        ┆   ┆        ┆        ┆ f64        ┆ ass       │
│            ┆            ┆            ┆            ┆   ┆        ┆        ┆            ┆ ---       │
│            ┆            ┆            ┆            ┆   ┆        ┆        ┆            ┆ f64       │
╞════════════╪════════════╪════════════╪════════════╪═══╪════════╪════════╪════════════╪═══════════╡
│ VUSAPWT1B1 ┆ A15        ┆ VUSAPWT1B1 ┆ VUSAPWT1B1 ┆ … ┆ 1      ┆ 1      ┆ null       ┆ null      │
│ r01_001    ┆            ┆ r01_001_00 ┆            ┆   ┆        ┆     

In [87]:
# write out results
scores.write_csv(f"{outputs_dir}/3_automated_qaqc_consensus_scores.csv")

In [88]:
# normalize scores using WT as reference
normalized_scores = scores.filter(pl.col("condition_name").is_in(scoring_conditions))

# First, sum adjusted scores across conditions for each variant
summed_scores = normalized_scores.group_by([
    'ad_orf_id', 'db_orf_id', 'db_mut_id', 'retest_batch'
]).agg(
    pl.col("adjusted_score").sum().alias("summed_score")
)

# Get WT scores (db_mut_id == 0) for normalization
wt_scores = summed_scores.filter(pl.col("db_mut_id") == 0).select([
    "ad_orf_id", "db_orf_id", "retest_batch", "summed_score"
]).rename({"summed_score": "wt_summed_score"})

# Join variant scores with their corresponding WT scores
normalized_scores = summed_scores.join(
    wt_scores, 
    on=["ad_orf_id", "db_orf_id", "retest_batch"], 
    how="left"
).with_columns(
    (pl.col("summed_score") / pl.col("wt_summed_score")).alias("normalized_score")
).filter(
    pl.col("wt_summed_score").is_not_null()  # Only keep variants that have a WT reference
)

In [89]:
normalized_scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,summed_score,wt_summed_score,normalized_score
i64,i64,i64,str,f64,f64,f64
2099,70497,3826,"""VUSAPWT1B1""",129.548044,589.147728,0.219891
1198,9634,0,"""VUSAPWT2B1""",683.401011,683.401011,1.0
12793,4506,0,"""VUSAPWT6B1""",0.0,0.0,NaN
71001,71001,0,"""VUSAPWT1B1""",1872.064783,1872.064783,1.0
54341,2713,200850,"""VUSAPWT1B1""",967.382818,868.328449,1.114075
…,…,…,…,…,…,…
2940,8416,216626,"""VUSAPWT1B2""",1.932789,0.0,inf
573,53030,0,"""VUSAPWT1B2""",2003.907939,2003.907939,1.0
6281,100016069,2635,"""VUSAPWT6B1""",121.117648,1711.277848,0.070776


In [90]:
# normalize scores using WT as reference
normalized_scores = scores.filter(pl.col("condition_name").is_in(scoring_conditions))

# First, sum adjusted scores across conditions for each variant
summed_scores = normalized_scores.group_by([
    'ad_orf_id', 'db_orf_id', 'db_mut_id', 'retest_batch'
]).agg(
    pl.col("adjusted_score").sum().alias("summed_adjusted_score")
)

# Get WT scores (db_mut_id == 0) for normalization
wt_scores = summed_scores.filter(pl.col("db_mut_id") == 0).select([
    "ad_orf_id", "db_orf_id", "retest_batch", "summed_adjusted_score"
]).rename({"summed_adjusted_score": "wt_summed_score"})

# Join variant scores with their corresponding WT scores
normalized_scores = summed_scores.join(
    wt_scores, 
    on=["ad_orf_id", "db_orf_id", "retest_batch"], 
    how="left"
).with_columns(
    (pl.col("summed_adjusted_score") / pl.col("wt_summed_score")).alias("normalized_score")
).filter(
    pl.col("wt_summed_score").is_not_null()  # Only keep variants that have a WT reference
).select([
    "ad_orf_id", "db_orf_id", "db_mut_id", "retest_batch", 
    "summed_adjusted_score", "wt_summed_score", "normalized_score"
]).rename({
    "summed_adjusted_score": "adjusted_score"  # Rename back to adjusted_score for consistency
})

normalized_scores = normalized_scores.with_columns(
    pl.col("retest_batch").str.extract(r"VUSAPWT(\d)B\d+", 1).alias("assay_id")
)

In [91]:
# cast all to same format
normalized_scores = normalized_scores.with_columns(
    pl.col("ad_orf_id").cast(pl.Float64).cast(pl.Int64).alias("ad_orf_id"),
    pl.col("db_orf_id").cast(pl.Float64).cast(pl.Int64).alias("db_orf_id"),
    pl.col("db_mut_id").cast(pl.Float64).cast(pl.Int64).alias("db_mut_id"),
    pl.col("assay_id").cast(pl.String).alias("assay_id"),
)

db_seq = db_seq.with_columns(
    pl.col("db_orf_id").cast(pl.Float64).cast(pl.Int64).alias("db_orf_id"),
    pl.col("db_mut_id").cast(pl.Float64).cast(pl.Int64).alias("db_mut_id"),
    pl.col("seq_db").cast(pl.String).alias("seq_db")
)

ad_seq = ad_seq.with_columns(
    pl.col("ad_orf_id").cast(pl.Float64).cast(pl.Int64).alias("ad_orf_id"),
    pl.col("seq_ad").cast(pl.String).alias("seq_ad"),
    pl.col("assay_id").cast(pl.String).alias("assay_id"),
)

normalized_scores = normalized_scores.join(ad_seq, on=["ad_orf_id", "assay_id"], how="left")
normalized_scores = normalized_scores.join(db_seq, on=["db_orf_id", "db_mut_id"], how="left")


# Filter to only keep interactions where both ad and db are confirmed
normalized_scores = normalized_scores.filter(
    (pl.col("seq_ad") == "1") & (pl.col("seq_db") == "1")
)
normalized_scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,adjusted_score,wt_summed_score,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class
i64,i64,i64,str,f64,f64,f64,str,str,str,f64,f64
3844,3844,0,"""VUSAPWT1B2""",466.0182,466.0182,1.0,"""1""","""1""","""1""",null,null
71963,2713,200818,"""VUSAPWT6B1""",2.34183,1848.130114,0.001267,"""6""","""1""","""1""",1.0,1.0
4935,4062,201428,"""VUSAPWT6B1""",609.266687,239.454258,2.544397,"""6""","""1""","""1""",1.0,1.0
4946,4062,201433,"""VUSAPWT1B2""",47.349674,1081.278893,0.04379,"""1""","""1""","""1""",1.0,1.0
54668,70497,3821,"""VUSAPWT1B1""",25.524183,295.218635,0.086459,"""1""","""1""","""1""",1.0,1.0
…,…,…,…,…,…,…,…,…,…,…,…
5923,71191,0,"""VUSAPWT6B1""",721.227259,721.227259,1.0,"""6""","""1""","""1""",null,null
3562,5066,0,"""VUSAPWT6B1""",16.840196,16.840196,1.0,"""6""","""1""","""1""",null,null
54341,2713,0,"""VUSAPWT1B1""",868.328449,868.328449,1.0,"""1""","""1""","""1""",null,null


In [92]:
# map ad_orf_id to ad_symbol
mapping_file = pd.read_csv(f"{inputs_dir}/cp_files/mapping_y2h.csv")
mapping_file.dropna(subset=['ad_orf_id'], inplace=True)
mapping_dict = mapping_file.set_index('ad_orf_id')['ad_symbol'].to_dict()

normalized_scores = normalized_scores.with_columns(
    pl.col("ad_orf_id").replace_strict(mapping_dict, default=None).alias("ad_symbol")
)
normalized_scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,adjusted_score,wt_summed_score,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol
i64,i64,i64,str,f64,f64,f64,str,str,str,f64,f64,str
3844,3844,0,"""VUSAPWT1B2""",466.0182,466.0182,1.0,"""1""","""1""","""1""",null,null,"""ACY1"""
71963,2713,200818,"""VUSAPWT6B1""",2.34183,1848.130114,0.001267,"""6""","""1""","""1""",1.0,1.0,"""MBD3L1"""
4935,4062,201428,"""VUSAPWT6B1""",609.266687,239.454258,2.544397,"""6""","""1""","""1""",1.0,1.0,"""SLC18A1"""
4946,4062,201433,"""VUSAPWT1B2""",47.349674,1081.278893,0.04379,"""1""","""1""","""1""",1.0,1.0,"""PTPRN"""
54668,70497,3821,"""VUSAPWT1B1""",25.524183,295.218635,0.086459,"""1""","""1""","""1""",1.0,1.0,"""KRT40"""
…,…,…,…,…,…,…,…,…,…,…,…,…
5923,71191,0,"""VUSAPWT6B1""",721.227259,721.227259,1.0,"""6""","""1""","""1""",null,null,"""ITGB3BP"""
3562,5066,0,"""VUSAPWT6B1""",16.840196,16.840196,1.0,"""6""","""1""","""1""",null,null,"""PDLIM7"""
54341,2713,0,"""VUSAPWT1B1""",868.328449,868.328449,1.0,"""1""","""1""","""1""",null,null,"""PTPN3"""


In [93]:
# Append metadata and write out file
metadata = pl.read_csv(f"{meta_outputs}/slim_metadata.csv").rename({
    "orf_id": "db_orf_id",
    "mut_id": "db_mut_id"
})

normalized_scores = normalized_scores.join(metadata, on=["db_orf_id", "db_mut_id"], how="left")
normalized_scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,adjusted_score,wt_summed_score,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
i64,i64,i64,str,f64,f64,f64,str,str,str,f64,f64,str,str,str,str,str,str,str,f64,f64,str
3844,3844,0,"""VUSAPWT1B2""",466.0182,466.0182,1.0,"""1""","""1""","""1""",null,null,"""ACY1""","""ACY1""",null,null,"""ENSG00000243989""",null,null,null,null,"""ACY1"""
71963,2713,200818,"""VUSAPWT6B1""",2.34183,1848.130114,0.001267,"""6""","""1""","""1""",1.0,1.0,"""MBD3L1""","""MLH1""","""Leu73Arg""","""218T>G""","""ENSG00000076242""","""CEGS2""","""1_Pathogenic""",null,2.0,"""MLH1_Leu73Arg"""
4935,4062,201428,"""VUSAPWT6B1""",609.266687,239.454258,2.544397,"""6""","""1""","""1""",1.0,1.0,"""SLC18A1""","""PLP1""","""Ile152Thr""","""455T>C""","""ENSG00000123560""","""CEGS2""","""1_Pathogenic""",null,2.0,"""PLP1_Ile152Thr"""
4946,4062,201433,"""VUSAPWT1B2""",47.349674,1081.278893,0.04379,"""1""","""1""","""1""",1.0,1.0,"""PTPRN""","""PLP1""","""Leu189Pro""","""566T>C""","""ENSG00000123560""","""CEGS2""","""4_VUS""",null,1.0,"""PLP1_Leu189Pro"""
54668,70497,3821,"""VUSAPWT1B1""",25.524183,295.218635,0.086459,"""1""","""1""","""1""",1.0,1.0,"""KRT40""","""AGXT""","""Phe152Ile""","""454T>A""","""ENSG00000172482""","""Edgotyping3""","""1_Pathogenic""",0.000103,2.0,"""AGXT_Phe152Ile"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
5923,71191,0,"""VUSAPWT6B1""",721.227259,721.227259,1.0,"""6""","""1""","""1""",null,null,"""ITGB3BP""","""SUOX""",null,null,"""ENSG00000139531""",null,null,null,null,"""SUOX"""
3562,5066,0,"""VUSAPWT6B1""",16.840196,16.840196,1.0,"""6""","""1""","""1""",null,null,"""PDLIM7""","""BAG3""",null,null,"""ENSG00000151929""",null,null,null,null,"""BAG3"""
54341,2713,0,"""VUSAPWT1B1""",868.328449,868.328449,1.0,"""1""","""1""","""1""",null,null,"""PTPN3""","""MLH1""",null,null,"""ENSG00000076242""",null,null,null,null,"""MLH1"""


In [94]:
normalized_scores.write_csv(f"{outputs_dir}/4_automated_normalized_consensus_scores.csv")

In [95]:
normalized_scores_pd = normalized_scores.to_pandas()
normalized_scores_pd

,ad_orf_id,db_orf_id,db_mut_id,retest_batch,adjusted_score,wt_summed_score,normalized_score,assay_id,seq_ad,seq_db,...,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
0,3844,3844,0,VUSAPWT1B2,466.018200,466.018200,1.000000,1,1,1,...,ACY1,ACY1,None,None,ENSG00000243989,None,None,NaN,NaN,ACY1
1,71963,2713,200818,VUSAPWT6B1,2.341830,1848.130114,0.001267,6,1,1,...,MBD3L1,MLH1,Leu73Arg,218T>G,ENSG00000076242,CEGS2,1_Pathogenic,NaN,2.0,MLH1_Leu73Arg
2,4935,4062,201428,VUSAPWT6B1,609.266687,239.454258,2.544397,6,1,1,...,SLC18A1,PLP1,Ile152Thr,455T>C,ENSG00000123560,CEGS2,1_Pathogenic,NaN,2.0,PLP1_Ile152Thr
3,4946,4062,201433,VUSAPWT1B2,47.349674,1081.278893,0.043790,1,1,1,...,PTPRN,PLP1,Leu189Pro,566T>C,ENSG00000123560,CEGS2,4_VUS,NaN,1.0,PLP1_Leu189Pro
4,54668,70497,3821,VUSAPWT1B1,25.524183,295.218635,0.086459,1,1,1,...,KRT40,AGXT,Phe152Ile,454T>A,ENSG00000172482,Edgotyping3,1_Pathogenic,0.000103,2.0,AGXT_Phe152Ile
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4095,5923,71191,0,VUSAPWT6B1,721.227259,721.227259,1.000000,6,1,1,...,ITGB3BP,SUOX,None,None,ENSG00000139531,None,None,NaN,NaN,SUOX
4096,3562,5066,0,VUSAPWT6B1,16.840196,16.840196,1.000000,6,1,1,...,PDLIM7,BAG3,None,None,ENSG00000151929,None,None,NaN,NaN,BAG3
4097,54341,2713,0,VUSAPWT1B1,868.328449,868.328449,1.000000,1,1,1,...,PTPN3,MLH1,None,None,ENSG00000076242,None,None,NaN,NaN,MLH1
4098,9625,71001,205676,VUSAPWT1B1,2116.997882,1606.828800,1.317501,1,1,1,...,MAGED1,UBQLN2,Pro509Ser,1525C>T,ENSG00000188021,CEGS2,4_VUS,0.000014,1.0,UBQLN2_Pro509Ser


In [97]:
normalized_scores_manual_pd = normalized_scores_manual.to_pandas()
normalized_scores_manual_pd[
    (normalized_scores_manual_pd.ad_orf_id == 4105) &
    (normalized_scores_manual_pd.db_orf_id == 4870) &
    (normalized_scores_manual_pd.db_mut_id == 215290)]

,ad_orf_id,db_orf_id,db_mut_id,retest_batch,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
1644,4105,4870,215290,VUSAPWT1B2,0.6875,1,1,1,1.0,1.0,KRT15,KRT6A,Leu468Pro,1403T>C,ENSG00000205420,CEGS2,5_Others,NaN,NaN,KRT6A_Leu468Pro


In [110]:
scores_pd = scores.to_pandas()
scores_pd[
    (scores_pd.ad_orf_id == '4105.0') &
    (scores_pd.db_orf_id == '4870.0') #&
    # (scores_pd.db_mut_id == 215290)
    ]

,ad_orf_id,ad_mut_id,db_orf_id,db_mut_id,condition_name,retest_batch,media_control,consensus_score,ad_symbol,symbol,...,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0,assay_id,db_group,db_condition_group,ppi_group
3532,4105.0,NaN,4870.0,215282.0,LWH25,VUSAPWT1B2,False,0.0,KRT15,KRT6A,...,ENSG00000205420,CEGS2,5_Others,NaN,NaN,KRT6A_Asn171Thr,1,4870.0_215282.0,4870.0_215282.0_LWH25_VUSAPWT1B2,4105.0_4870.0_215282.0_VUSAPWT1B2
3840,4105.0,NaN,4870.0,215289.0,LWH25,VUSAPWT1B2,False,2.0,KRT15,KRT6A,...,ENSG00000205420,CEGS2,1_Pathogenic,NaN,2.0,KRT6A_Ile462Asn,1,4870.0_215289.0,4870.0_215289.0_LWH25_VUSAPWT1B2,4105.0_4870.0_215289.0_VUSAPWT1B2
3927,4105.0,NaN,4870.0,215279.0,LWH1,VUSAPWT1B2,False,3.0,KRT15,KRT6A,...,ENSG00000205420,CEGS2,5_Others,NaN,NaN,KRT6A_Ile178Asn,1,4870.0_215279.0,4870.0_215279.0_LWH1_VUSAPWT1B2,4105.0_4870.0_215279.0_VUSAPWT1B2
4639,4105.0,NaN,4870.0,215293.0,LWH1,VUSAPWT1B2,False,4.0,KRT15,KRT6A,...,ENSG00000205420,CEGS2,1_Pathogenic,NaN,1.0,KRT6A_Tyr465His,1,4870.0_215293.0,4870.0_215293.0_LWH1_VUSAPWT1B2,4105.0_4870.0_215293.0_VUSAPWT1B2
6641,4105.0,NaN,4870.0,215272.0,LWH1,VUSAPWT1B2,False,4.0,KRT15,KRT6A,...,ENSG00000205420,CEGS2,1_Pathogenic,NaN,1.0,KRT6A_Leu170Phe,1,4870.0_215272.0,4870.0_215272.0_LWH1_VUSAPWT1B2,4105.0_4870.0_215272.0_VUSAPWT1B2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69577,4105.0,NaN,4870.0,215279.0,LW,VUSAPWT1B2,False,4.0,KRT15,KRT6A,...,ENSG00000205420,CEGS2,5_Others,NaN,NaN,KRT6A_Ile178Asn,1,4870.0_215279.0,4870.0_215279.0_LW_VUSAPWT1B2,4105.0_4870.0_215279.0_VUSAPWT1B2
70155,4105.0,NaN,4870.0,215276.0,LWAH1,VUSAPWT1B2,False,3.0,KRT15,KRT6A,...,ENSG00000205420,CEGS2,5_Others,NaN,NaN,KRT6A_Arg164Pro,1,4870.0_215276.0,4870.0_215276.0_LWAH1_VUSAPWT1B2,4105.0_4870.0_215276.0_VUSAPWT1B2
70596,4105.0,NaN,4870.0,215281.0,LWA,VUSAPWT1B2,False,3.0,KRT15,KRT6A,...,ENSG00000205420,CEGS2,5_Others,NaN,NaN,KRT6A_Ile167Ser,1,4870.0_215281.0,4870.0_215281.0_LWA_VUSAPWT1B2,4105.0_4870.0_215281.0_VUSAPWT1B2
70925,4105.0,NaN,4870.0,215281.0,LWH10,VUSAPWT1B2,False,3.0,KRT15,KRT6A,...,ENSG00000205420,CEGS2,5_Others,NaN,NaN,KRT6A_Ile167Ser,1,4870.0_215281.0,4870.0_215281.0_LWH10_VUSAPWT1B2,4105.0_4870.0_215281.0_VUSAPWT1B2


In [114]:
normalized_scores_manual_pd[
    (normalized_scores_manual_pd.ad_orf_id == 4105.0) &
    (normalized_scores_manual_pd.db_orf_id == 4870.0) #&
    # (scores_pd.db_mut_id == 215290)
    ]

,ad_orf_id,db_orf_id,db_mut_id,retest_batch,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
333,4105,4870,215286,VUSAPWT1B2,0.6875,1,1,1,1.0,1.0,KRT15,KRT6A,Glu472Lys,1414G>A,ENSG00000205420,CEGS2,1_Pathogenic,NaN,2.0,KRT6A_Glu472Lys
633,4105,4870,215298,VUSAPWT1B2,0.8750,1,1,1,1.0,1.0,KRT15,KRT6A,Glu472Asp,1416G>C,ENSG00000205420,CEGS2,5_Others,NaN,NaN,KRT6A_Glu472Asp
766,4105,4870,215273,VUSAPWT1B2,0.7500,1,1,1,1.0,1.0,KRT15,KRT6A,Ser176Pro,526T>C,ENSG00000205420,CEGS2,5_Others,NaN,NaN,KRT6A_Ser176Pro
789,4105,4870,215270,VUSAPWT1B2,0.6250,1,1,1,1.0,1.0,KRT15,KRT6A,Asn171Ser,512A>G,ENSG00000205420,CEGS2,1_Pathogenic,NaN,2.0,KRT6A_Asn171Ser
796,4105,4870,0,VUSAPWT1B2,0.0000,1,1,1,NaN,NaN,KRT15,KRT6A,None,None,ENSG00000205420,None,None,NaN,NaN,KRT6A
830,4105,4870,215281,VUSAPWT1B2,0.7500,1,1,1,1.0,1.0,KRT15,KRT6A,Ile167Ser,500T>G,ENSG00000205420,CEGS2,5_Others,NaN,NaN,KRT6A_Ile167Ser
1135,4105,4870,215293,VUSAPWT1B2,0.7500,1,1,1,1.0,1.0,KRT15,KRT6A,Tyr465His,1393T>C,ENSG00000205420,CEGS2,1_Pathogenic,NaN,1.0,KRT6A_Tyr465His
1433,4105,4870,215280,VUSAPWT1B2,0.4375,1,1,1,1.0,1.0,KRT15,KRT6A,Glu163Lys,487G>A,ENSG00000205420,CEGS2,1_Pathogenic,NaN,1.0,KRT6A_Glu163Lys
1459,4105,4870,215274,VUSAPWT1B2,0.8750,1,1,1,1.0,1.0,KRT15,KRT6A,Asn171Asp,511A>G,ENSG00000205420,CEGS2,1_Pathogenic,NaN,1.0,KRT6A_Asn171Asp
1466,4105,4870,215272,VUSAPWT1B2,0.6875,1,1,1,1.0,2.0,KRT15,KRT6A,Leu170Phe,508C>T,ENSG00000205420,CEGS2,1_Pathogenic,NaN,1.0,KRT6A_Leu170Phe
